# Private CUDA TCN: metadata and synthetic smoke

Keep this notebook private. Root has approved the private dataset/GPU workflow. This notebook defaults to metadata plus synthetic-only smoke; it does not train either real fold. No package download/install is performed. Actual hosted PyTorch/CUDA compatibility, memory and runtime are unverified until the smoke passes. The current public leaderboard score is never used for hyperparameter tuning.


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys
TRAINER_TEXT = '"""Portable, gated CUDA TCN experiment; no installation or network operations.\n\nDefault mode is a synthetic-only CUDA smoke. Real fitting requires an explicit\nsingle year, a successful matching smoke, a verified input bundle, and (for2022)\nan externally produced2021 selection lock. No command trains both folds.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport platform\nimport random\nimport shlex\nimport shutil\nimport stat\nimport statistics\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom typing import Any\nfrom unittest.mock import patch\nimport zipfile\n\nfor key in [\'OMP_NUM_THREADS\', \'OPENBLAS_NUM_THREADS\', \'MKL_NUM_THREADS\', \'VECLIB_MAXIMUM_THREADS\']:\n    os.environ[key] = \'2\'\nos.environ[\'CUBLAS_WORKSPACE_CONFIG\'] = \':4096:8\'\nsys.dont_write_bytecode = True\nSEED = 4173\nFEATURES = 395\nHOURS = 168\nBATCH = 16\nEPOCHS = 24\n\n\ndef require(condition: bool, message: str) -> None:\n    """Reject incomplete prerequisites rather than silently changing the run."""\n    if not condition:\n        raise ValueError(message)\n\n\ndef digest(path: Path) -> str:\n    """Return a streaming SHA256."""\n    value = hashlib.sha256()\n    with path.open(\'rb\') as handle:\n        for block in iter(lambda: handle.read(1 << 20), b\'\'):\n            value.update(block)\n    return value.hexdigest()\n\n\ndef safe_relative(name: str) -> Path:\n    """Permit only plain relative POSIX paths inside a payload root."""\n    path = PurePosixPath(name)\n    require(bool(name) and not path.is_absolute() and \'..\' not in path.parts and \'\\\\\' not in name,\n            f\'Unsafe payload path: {name}\')\n    require(str(path) == name and name != \'.\', f\'Noncanonical payload path: {name}\')\n    return Path(*path.parts)\n\n\ndef stage_bundle(source: Path, destination: Path, manifest_sha256: str,\n                 archive_sha256: str | None = None) -> dict:\n    """Verify an exact payload allowlist before copying to a separate work root."""\n    source, destination = source.resolve(), destination.resolve()\n    require(source != destination and (not source.is_dir() or not destination.is_relative_to(source)),\n            \'Stage destination must be separate from the read-only input\')\n    zipped = source.is_file()\n    if zipped:\n        if archive_sha256 is not None:\n            require(digest(source) == archive_sha256, \'Input archive checksum mismatch\')\n        with zipfile.ZipFile(source) as archive:\n            infos = archive.infolist()\n            require(len({item.filename for item in infos}) == len(infos), \'Duplicate zip members\')\n            manifest_bytes = archive.read(\'bundle_manifest.json\')\n    else:\n        manifest_bytes = (source / \'bundle_manifest.json\').read_bytes()\n    require(hashlib.sha256(manifest_bytes).hexdigest() == manifest_sha256, \'Input bundle manifest checksum mismatch\')\n    manifest = json.loads(manifest_bytes)\n    entries = manifest[\'payload_files\']\n    names = [item[\'path\'] for item in entries]\n    require(len(names) == len(set(names)) and \'bundle_manifest.json\' not in names, \'Invalid bundle allowlist\')\n    for name in names:\n        safe_relative(name)\n    allowed = set(names) | {\'bundle_manifest.json\'}\n    reuse = destination.exists() and any(destination.iterdir())\n    if reuse:\n        found = {str(path.relative_to(destination)) for path in destination.rglob(\'*\') if path.is_file()}\n        require(found == allowed and not any(path.is_symlink() for path in destination.rglob(\'*\')), \'Existing stage has unexpected files or symlinks\')\n    destination.mkdir(parents=True, exist_ok=True)\n    if zipped:\n        with zipfile.ZipFile(source) as archive:\n            require(set(archive.namelist()) == allowed, \'Archive contains missing or unlisted members\')\n            for item in archive.infolist():\n                require(not item.is_dir() and not stat.S_ISLNK(item.external_attr >> 16), \'Archive directories/symlinks are forbidden\')\n                if not reuse:\n                    target = destination / safe_relative(item.filename)\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    with archive.open(item) as incoming, target.open(\'xb\') as outgoing:\n                        shutil.copyfileobj(incoming, outgoing)\n    else:\n        found = {str(path.relative_to(source)) for path in source.rglob(\'*\') if path.is_file()}\n        require(found == allowed, \'Input directory contains missing or unlisted files\')\n        require(not any(path.is_symlink() for path in source.rglob(\'*\')), \'Input symlinks are forbidden\')\n        if not reuse:\n            for name in sorted(allowed):\n                target = destination / safe_relative(name)\n                target.parent.mkdir(parents=True, exist_ok=True)\n                shutil.copyfile(source / name, target)\n    for item in entries:\n        target = destination / safe_relative(item[\'path\'])\n        require(target.stat().st_size == item[\'size_bytes\'] and digest(target) == item[\'sha256\'],\n                f\'Staged payload checksum/size mismatch: {item["path"]}\')\n    require(digest(destination / \'bundle_manifest.json\') == manifest_sha256, \'Staged manifest changed\')\n    return manifest\n\n\ndef read_targets_through_year(path: Path, year: int, read_csv: Any) -> Any:\n    """Read all case metadata but decode targets only through the selected year."""\n    import numpy as np\n    import pandas as pd\n    meta = read_csv(path, usecols=[\'Run No\', \'starttime\'])\n    dates = pd.to_datetime(meta.starttime, utc=True)\n    allowed = dates < pd.Timestamp(f\'{year + 1}-01-01\', tz=\'UTC\')\n    count = int(allowed.sum())\n    require(np.array_equal(allowed.to_numpy(), np.arange(len(meta)) < count), \'Expected chronological target prefix\')\n    prior = read_csv(path, nrows=count)\n    later = pd.DataFrame(np.zeros((len(meta) - count, len(prior.columns) - 3), dtype=np.uint8), columns=prior.columns[3:])\n    later.insert(0, \'total_calculation_time\', 0.)\n    later.insert(0, \'starttime\', meta.starttime.iloc[count:].to_numpy())\n    later.insert(0, \'Run No\', meta[\'Run No\'].iloc[count:].to_numpy())\n    return pd.concat([prior, later], ignore_index=True)\n\n\ndef training_stats(cube: Any, case_indices: Any) -> tuple[Any, Any]:\n    """Streaming float64 population moments from retained training cases only."""\n    import numpy as np\n    count = 0\n    mean = np.zeros(cube.shape[2], dtype=np.float64)\n    m2 = np.zeros_like(mean)\n    for offset in range(0, len(case_indices), 32):\n        block = cube[case_indices[offset:offset + 32]].reshape(-1, cube.shape[2]).astype(np.float64)\n        require(np.isfinite(block).all(), \'Nonfinite training features\')\n        n = len(block)\n        local_mean = block.mean(axis=0)\n        local_m2 = np.sum((block - local_mean) ** 2, axis=0)\n        delta = local_mean - mean\n        new_count = count + n\n        m2 += local_m2 + delta ** 2 * count * n / new_count\n        mean += delta * n / new_count\n        count = new_count\n    require(count > 0, \'No retained training observations\')\n    scale = np.sqrt(np.maximum(m2 / count, 0.))\n    scale[scale < 1e-6] = 1.\n    return mean, scale\n\n\ndef normalized_batch(cube: Any, indices: Any, mean: Any, scale: Any) -> Any:\n    """Normalize whole cases, clip, and return contiguous float32 N,C,T input."""\n    import numpy as np\n    value = np.clip((cube[indices].astype(np.float64) - mean) / scale, -10., 10.).astype(np.float32)\n    require(np.isfinite(value).all(), \'Nonfinite normalized batch\')\n    return np.ascontiguousarray(value.transpose(0, 2, 1))\n\n\ndef build_model(torch: Any, features: int = FEATURES) -> Any:\n    """Build the unchanged64-channel centered61-hour residual TCN architecture."""\n    nn = torch.nn\n\n    class HourNorm(nn.Module):\n        def __init__(self) -> None:\n            super().__init__()\n            self.norm = nn.LayerNorm(64)\n\n        def forward(self, x: Any) -> Any:\n            return self.norm(x.transpose(1, 2)).transpose(1, 2)\n\n    class ResidualBlock(nn.Module):\n        def __init__(self, dilation: int) -> None:\n            super().__init__()\n            layers = []\n            for _ in range(2):\n                layers.extend([nn.Conv1d(64, 64, 3, padding=dilation, dilation=dilation, padding_mode=\'replicate\'),\n                               HourNorm(), nn.ReLU(), nn.Dropout(.1)])\n            self.layers = nn.Sequential(*layers)\n\n        def forward(self, x: Any) -> Any:\n            return torch.relu(x + self.layers(x))\n\n    return nn.Sequential(nn.Conv1d(features, 64, 1), nn.ReLU(),\n                         *[ResidualBlock(d) for d in [1, 2, 4, 8]], nn.Conv1d(64, 14, 1))\n\n\ndef cuda_environment(torch: Any) -> dict:\n    """Require genuine CUDA and record all visible devices; use cuda:0 only."""\n    require(torch.cuda.is_available(), \'CUDA is required; CPU fallback is forbidden\')\n    require(torch.version.cuda is not None, \'PyTorch was built without CUDA\')\n    torch.set_num_threads(2)\n    torch.set_num_interop_threads(1)\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    random.seed(SEED)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cuda.matmul.allow_tf32 = False\n    devices = []\n    for index in range(torch.cuda.device_count()):\n        prop = torch.cuda.get_device_properties(index)\n        devices.append({\'index\': index, \'name\': prop.name, \'total_memory_bytes\': prop.total_memory,\n                        \'capability\': [prop.major, prop.minor]})\n    return {\'torch\': torch.__version__, \'cuda_build\': torch.version.cuda,\n            \'cudnn\': torch.backends.cudnn.version(), \'devices\': devices, \'selected_device\': \'cuda:0\',\n            \'deterministic_algorithms\': True, \'float32\': True, \'amp\': False, \'compile\': False,\n            \'cpu_fallback\': False, \'CUBLAS_WORKSPACE_CONFIG\': os.environ[\'CUBLAS_WORKSPACE_CONFIG\']}\n\n\ndef synthetic_smoke(torch: Any) -> dict:\n    """Measure only synthetic full-case updates; no supplied input table is read."""\n    model = build_model(torch).to(\'cuda:0\').train()\n    parameter_count = sum(parameter.numel() for parameter in model.parameters())\n    require(parameter_count == 126094, \'TCN parameter count changed\')\n    optimizer = torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=.0001, foreach=False)\n    loss_fn = torch.nn.BCEWithLogitsLoss()\n    x = torch.randn(BATCH, FEATURES, HOURS)\n    y = torch.randint(0, 2, (BATCH, 14, HOURS)).float()\n    times, losses = [], []\n    for step in range(12):\n        if step == 2:\n            torch.cuda.reset_peak_memory_stats(0)\n        torch.cuda.synchronize(0)\n        start = time.perf_counter()\n        optimizer.zero_grad(set_to_none=True)\n        logits = model(x.to(\'cuda:0\'))\n        require(tuple(logits.shape) == (BATCH, 14, HOURS) and logits.device.type == \'cuda\', \'Synthetic output shape/device mismatch\')\n        loss = loss_fn(logits, y.to(\'cuda:0\'))\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n        optimizer.step()\n        torch.cuda.synchronize(0)\n        elapsed = time.perf_counter() - start\n        require(torch.isfinite(loss).item(), \'Synthetic loss is nonfinite\')\n        losses.append(float(loss.detach().cpu()))\n        if step >= 2:\n            times.append(elapsed)\n    return {\'synthetic_only\': True, \'input_shape\': [BATCH, FEATURES, HOURS], \'warmup_steps\': 2,\n            \'timed_steps\': 10, \'step_seconds\': times, \'median_step_seconds\': statistics.median(times),\n            \'max_allocated_bytes\': torch.cuda.max_memory_allocated(0),\n            \'max_reserved_bytes\': torch.cuda.max_memory_reserved(0), \'synthetic_losses\': losses,\n            \'parameters\': parameter_count, \'output_shape\': [BATCH, 14, HOURS],\n            \'timing_scope\': \'host-to-device transfer, forward, backward, clipping, optimizer, synchronization; excludes real feature construction and loading\'}\n\n\ndef validate_selection_lock(path: Path, expected_sha: str, protocol: dict, protocol_sha: str) -> dict:\n    """Require a locally reviewed2021 choice before the separate2022 command."""\n    require(len(expected_sha) == 64 and digest(path) == expected_sha, \'Explicit2021 selection-lock SHA256 mismatch\')\n    locked = json.loads(path.read_text())\n    require(locked[\'version\'] == \'cuda_tcn_2021_blend_lock_v1\' and locked[\'selection_year\'] == 2021,\n            \'Unexpected selection-lock schema/year\')\n    require(locked[\'training_protocol_sha256\'] == protocol_sha, \'Selection lock uses another training protocol\')\n    require(locked[\'anchor_run\'] == protocol[\'anchor\'][\'run_id\']\n            and locked[\'anchor_sha256\'] == protocol[\'anchor\'][\'analysis_sha256\'], \'Selection lock uses another anchor\')\n    require(locked[\'candidate_neural_weights\'] == [0., .1, .25, .5]\n            and locked[\'selected_neural_weight\'] in locked[\'candidate_neural_weights\'], \'Unexpected blend selection\')\n    require(locked.get(\'training_2021_complete\') is True, \'2021 fitting/prediction completion is not recorded\')\n    for key in [\'neural_run_manifest_sha256\', \'neural_validation_2021_sha256\']:\n        require(isinstance(locked[key], str) and len(locked[key]) == 64\n                and all(c in \'0123456789abcdef\' for c in locked[key]), \'Missing2021 neural provenance hash\')\n    return locked\n\n\ndef build_verified_features(stage: Path, year: int, protocol: dict) -> tuple:\n    """Import only staged frozen code and redact target rows after selected year."""\n    import numpy as np\n    import pandas as pd\n    for name, module in list(sys.modules.items()):\n        if name.startswith(\'sintef_project\') and getattr(module, \'__file__\', None):\n            require(Path(module.__file__).resolve().is_relative_to(stage), \'A foreign sintef_project module is already imported; use a fresh process\')\n    sys.path.insert(0, str(stage))\n    from sintef_project.features import build_features\n    from sintef_project.physics import add_physics_features\n    from sintef_project.context_features import add_context_features\n    from sintef_project.lp_features import add_lp_features\n    from sintef_project.normalized import add_normalized_features\n    target = (stage / \'sintef_project/data/kernel/Unit_commitment_decisions.csv\').resolve()\n    original_read = pd.read_csv\n\n    def guarded_read(path: Any, *args: Any, **kwargs: Any) -> Any:\n        if isinstance(path, (str, Path)) and Path(path).resolve() == target:\n            if kwargs.get(\'usecols\') == [\'Run No\', \'starttime\']:\n                return original_read(path, *args, **kwargs)\n            require(not args and not kwargs, \'Frozen full-target read signature changed\')\n            return read_targets_through_year(target, year, original_read)\n        return original_read(path, *args, **kwargs)\n\n    with patch.object(pd, \'read_csv\', guarded_read):\n        x, labels, train, mapping, meta = build_features(stage)\n        x, physics = add_physics_features(x, stage)\n        x, context = add_context_features(x, stage)\n        x, lp = add_lp_features(x, stage)\n        x, normalized = add_normalized_features(x, stage)\n    require(x.columns.tolist() == protocol[\'feature_columns\'] and x.shape[1] == FEATURES, \'Frozen395 feature order mismatch\')\n    require(np.isfinite(x.to_numpy()).all(), \'Nonfinite real features\')\n    meta[\'feature_spec\'].update(physics=physics, context=context, lp=lp, normalized=normalized)\n    baseline = json.loads((stage / \'artifacts/lgb-normalized-lp-001/manifest.json\').read_text())\n    expected_spec = json.loads(json.dumps(baseline[\'feature_spec\']))\n    actual_spec = json.loads(json.dumps(meta[\'feature_spec\']))\n    expected_spec[\'lp\'].pop(\'cache_directory\')\n    actual_spec[\'lp\'].pop(\'cache_directory\')\n    require(expected_spec == actual_spec, \'Frozen feature formulas changed beyond portable cache-root path\')\n    return x, labels, train, mapping, meta, baseline\n\n\ndef train_fold(torch: Any, stage: Path, year: int, protocol: dict, protocol_sha: str,\n               out: Path, report: dict) -> None:\n    """Train exactly24 epochs on complete purged cases; evaluate only afterward."""\n    import numpy as np\n    import pandas as pd\n    from sklearn.metrics import roc_auc_score\n    x, labels, train, mapping, meta, baseline = build_verified_features(stage, year, protocol)\n    cube = x.to_numpy(dtype=np.float32).reshape(len(train) + len(mapping), HOURS, FEATURES)\n    del x\n    dates = pd.to_datetime(train.starttime, utc=True)\n    begin = pd.Timestamp(f\'{year}-01-01\', tz=\'UTC\')\n    end = pd.Timestamp(f\'{year + 1}-01-01\', tz=\'UTC\')\n    tr = np.flatnonzero(dates < begin - pd.Timedelta(days=6))\n    va = np.flatnonzero((dates >= begin) & (dates < end))\n    require(dates.iloc[tr].max() + pd.Timedelta(hours=167) < begin, \'Training targets overlap validation\')\n    require(train.iloc[tr][\'Run No\'].tolist() == baseline[\'folds\'][str(year)][\'train_ids\']\n            and train.iloc[va][\'Run No\'].tolist() == baseline[\'folds\'][str(year)][\'validation_ids\'], \'Purged IDs differ from frozen baseline\')\n    units = meta[\'units\']\n    y = np.column_stack([labels[unit] for unit in units]).reshape(len(train), HOURS, 14)\n    del labels\n    mean, scale = training_stats(cube, tr)\n    np.savez_compressed(out / \'normalization.npz\', mean=mean, scale=scale, columns=protocol[\'feature_columns\'], training_case_ids=train.iloc[tr][\'Run No\'].to_numpy())\n    report.update(feature_spec=meta[\'feature_spec\'], units=units, train_ids=train.iloc[tr][\'Run No\'].tolist(),\n                  validation_ids=train.iloc[va][\'Run No\'].tolist(), purge_days=6,\n                  all_training_case_hours=True, training_rows=len(tr) * HOURS,\n                  validation_label_scope=f\'Only case starts in{year}; no later target rows decoded\',\n                  baseline_reference_sha256=digest(stage / \'artifacts/lgb-normalized-lp-001/manifest.json\'))\n    model = build_model(torch).to(\'cuda:0\')\n    optimizer = torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=.0001, betas=(.9, .999), eps=1e-8, foreach=False)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=.0001)\n    loss_fn = torch.nn.BCEWithLogitsLoss()\n    rng = np.random.default_rng(SEED)\n    orders, epochs = [], []\n    for epoch in range(EPOCHS):\n        model.train()\n        order = rng.permutation(tr)\n        orders.append(train.iloc[order][\'Run No\'].to_numpy())\n        started = time.perf_counter()\n        loss_sum = 0.\n        learning_rate = optimizer.param_groups[0][\'lr\']\n        for offset in range(0, len(order), BATCH):\n            indices = order[offset:offset + BATCH]\n            inputs = torch.from_numpy(normalized_batch(cube, indices, mean, scale)).to(\'cuda:0\')\n            targets = torch.from_numpy(np.ascontiguousarray(y[indices].transpose(0, 2, 1)).astype(np.float32)).to(\'cuda:0\')\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(inputs), targets)\n            require(torch.isfinite(loss).item(), \'Nonfinite training loss\')\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n            optimizer.step()\n            loss_sum += float(loss.detach().cpu()) * len(indices)\n        scheduler.step()\n        torch.cuda.synchronize(0)\n        epochs.append({\'epoch\': epoch + 1, \'training_bce\': loss_sum / len(tr),\n                       \'learning_rate\': learning_rate, \'seconds\': time.perf_counter() - started})\n        print(json.dumps(epochs[-1]), flush=True)\n    np.save(out / \'epoch_training_case_orders.npy\', np.stack(orders))\n\n    def cpu_tree(value: Any) -> Any:\n        if torch.is_tensor(value):\n            return value.detach().cpu()\n        if isinstance(value, dict):\n            return {key: cpu_tree(item) for key, item in value.items()}\n        if isinstance(value, (list, tuple)):\n            return type(value)(cpu_tree(item) for item in value)\n        return value\n\n    checkpoint = {\'model_state_dict\': cpu_tree(model.state_dict()), \'optimizer_state_dict\': cpu_tree(optimizer.state_dict()),\n                  \'scheduler_state_dict\': scheduler.state_dict(), \'epoch\': EPOCHS, \'seed\': SEED,\n                  \'protocol_sha256\': protocol_sha, \'features\': protocol[\'feature_columns\'], \'units\': units,\n                  \'trainer_sha256\': digest(Path(__file__)), \'bundle_manifest_sha256\': protocol[\'bundle_manifest_sha256\'],\n                  \'normalization_sha256\': digest(out / \'normalization.npz\'),\n                  \'numpy_rng_state\': rng.bit_generator.state, \'torch_rng_cpu\': torch.get_rng_state(),\n                  \'torch_rng_cuda\': [value.cpu() for value in torch.cuda.get_rng_state_all()]}\n    torch.save(checkpoint, out / f\'tcn_{year}.pt\')\n    # No validation evaluation or checkpoint choice was made during training.\n    model.eval()\n    predictions = np.empty((len(va), HOURS, 14), dtype=np.float32)\n    with torch.inference_mode():\n        for offset in range(0, len(va), BATCH):\n            indices = va[offset:offset + BATCH]\n            inputs = torch.from_numpy(normalized_batch(cube, indices, mean, scale)).to(\'cuda:0\')\n            probabilities = torch.sigmoid(model(inputs)).cpu().numpy().transpose(0, 2, 1)\n            predictions[offset:offset + len(indices)] = probabilities\n    prediction = predictions.reshape(-1, 14)\n    truth = y[va].reshape(-1, 14)\n    require(np.isfinite(prediction).all() and prediction.min() >= 0 and prediction.max() <= 1, \'Invalid OOF probabilities\')\n    np.savez_compressed(out / f\'validation_{year}.npz\', predictions=prediction, labels=truth,\n                        case_ids=train.iloc[va][\'Run No\'].to_numpy(), units=units)\n    # Fresh model + saved normalizer replay; trusted hash checked before loading.\n    checkpoint_path = out / f\'tcn_{year}.pt\'\n    checkpoint_hash = digest(checkpoint_path)\n    loaded = torch.load(checkpoint_path, map_location=\'cpu\', weights_only=True)\n    require(digest(checkpoint_path) == checkpoint_hash, \'Checkpoint changed during trusted load\')\n    for key, expected in [(\'protocol_sha256\', protocol_sha), (\'trainer_sha256\', digest(Path(__file__))),\n                          (\'bundle_manifest_sha256\', protocol[\'bundle_manifest_sha256\']),\n                          (\'features\', protocol[\'feature_columns\']), (\'units\', units), (\'epoch\', EPOCHS), (\'seed\', SEED)]:\n        require(loaded[key] == expected, f\'Checkpoint replay metadata mismatch: {key}\')\n    require(loaded[\'normalization_sha256\'] == digest(out / \'normalization.npz\'), \'Saved normalizer changed\')\n    with np.load(out / \'normalization.npz\', allow_pickle=False) as saved_stats:\n        require(saved_stats[\'columns\'].tolist() == protocol[\'feature_columns\']\n                and np.array_equal(saved_stats[\'training_case_ids\'], train.iloc[tr][\'Run No\'].to_numpy()), \'Normalizer column/training scope changed\')\n        saved_mean, saved_scale = saved_stats[\'mean\'], saved_stats[\'scale\']\n        require(saved_mean.dtype == saved_scale.dtype == np.float64, \'Normalizer must preserve float64 statistics\')\n    fresh = build_model(torch).to(\'cuda:0\')\n    fresh.load_state_dict(loaded[\'model_state_dict\'], strict=True)\n    fresh.eval()\n    repeated = np.empty_like(predictions)\n    with torch.inference_mode():\n        for offset in range(0, len(va), BATCH):\n            indices = va[offset:offset + BATCH]\n            inputs = torch.from_numpy(normalized_batch(cube, indices, saved_mean, saved_scale)).to(\'cuda:0\')\n            repeated[offset:offset + len(indices)] = torch.sigmoid(fresh(inputs)).cpu().numpy().transpose(0, 2, 1)\n    replay_error = float(np.max(np.abs(repeated - predictions)))\n    require(repeated.tobytes() == predictions.tobytes(), \'Fresh checkpoint OOF replay is not byte-identical\')\n    np.save(out / f\'replay_predictions_{year}.npy\', repeated.reshape(-1, 14))\n    report.update(epochs=epochs, completed_epochs=EPOCHS,\n                  raw_micro_auc=float(roc_auc_score(truth.ravel(), prediction.ravel())),\n                  unit_auc={unit: float(roc_auc_score(truth[:, j], prediction[:, j])) for j, unit in enumerate(units)},\n                  max_cuda_allocated_bytes=torch.cuda.max_memory_allocated(0),\n                  max_cuda_reserved_bytes=torch.cuda.max_memory_reserved(0),\n                  checkpoint_sha256=checkpoint_hash,\n                  validation_sha256=digest(out / f\'validation_{year}.npz\'),\n                  checkpoint_replay={\'fresh_model\': True, \'saved_float64_normalizer\': True, \'eval_inference_mode\': True,\n                                     \'bitwise_equal\': True, \'max_abs_probability_difference\': replay_error,\n                                     \'checkpoint_load\': \'hash-verified own checkpoint, weights_only=True, map_location=cpu\',\n                                     \'replay_predictions_sha256\': digest(out / f\'replay_predictions_{year}.npy\')},\n                  postprocessing=\'none; raw14-unit probabilities for local frozen blend selection\')\n    print(\'FOLD\', year, \'raw_micro_auc\', report[\'raw_micro_auc\'], flush=True)\n\n\ndef main() -> None:\n    """Run either a synthetic CUDA check or one explicitly selected fold."""\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--mode\', choices=[\'smoke\', \'fit\'], default=\'smoke\')\n    parser.add_argument(\'--year\', type=int, choices=[2021, 2022])\n    parser.add_argument(\'--protocol\', type=Path, required=True)\n    parser.add_argument(\'--run-id\', required=True)\n    parser.add_argument(\'--output-root\', type=Path, default=Path(\'/kaggle/working/tcn-runs\'))\n    parser.add_argument(\'--bundle\', type=Path)\n    parser.add_argument(\'--stage-root\', type=Path, default=Path(\'/kaggle/working/tdt4173-tcn-inputs\'))\n    parser.add_argument(\'--smoke-manifest\', type=Path)\n    parser.add_argument(\'--selection-lock\', type=Path)\n    parser.add_argument(\'--selection-lock-sha256\')\n    args = parser.parse_args()\n    out = args.output_root / args.run_id\n    out.mkdir(parents=True, exist_ok=False)\n    started = time.time()\n    report = {\'timestamp_utc\': datetime.now(timezone.utc).isoformat(), \'run_id\': args.run_id,\n              \'command\': shlex.join([sys.executable, *sys.argv]), \'mode\': args.mode, \'year\': args.year,\n              \'seed\': SEED, \'status\': \'initializing\', \'real_feature_table_loaded\': False,\n              \'early_stopping\': None, \'checkpoint_selection\': \'fixed final epoch24 only\',\n              \'environment\': {\'python\': sys.version, \'platform\': platform.platform(), \'packages\': {}},\n              \'kaggle_submission\': None, \'blend_selection\': \'local only, after downloading OOF; no anchor arrays uploaded\'}\n    try:\n        protocol_sha = digest(args.protocol)\n        protocol = json.loads(args.protocol.read_text())\n        require(protocol[\'version\'] == \'cuda_tcn_v2_joint_anchor\', \'Unexpected training protocol\')\n        require(protocol[\'trainer_sha256\'] == digest(Path(__file__)), \'Trainer does not match frozen protocol hash\')\n        require(len(protocol[\'feature_columns\']) == FEATURES and protocol[\'epochs\'] == EPOCHS\n                and protocol[\'batch_cases\'] == BATCH and protocol[\'seed\'] == SEED, \'Protocol constants changed\')\n        report[\'protocol_sha256\'] = protocol_sha\n        report[\'trainer_sha256\'] = digest(Path(__file__))\n        if args.mode == \'fit\':\n            require(args.year is not None and args.bundle is not None and args.smoke_manifest is not None,\n                    \'Fit requires one explicit year, input bundle and completed synthetic smoke\')\n            if args.year == 2022:\n                require(args.selection_lock is not None and args.selection_lock_sha256 is not None,\n                        \'2022 requires the externally reviewed local2021 selection lock and explicitSHA256\')\n                report[\'selection_lock\'] = validate_selection_lock(args.selection_lock, args.selection_lock_sha256, protocol, protocol_sha)\n                report[\'selection_lock_sha256\'] = digest(args.selection_lock)\n            else:\n                require(args.selection_lock is None, \'2021 must precede external blend selection\')\n        else:\n            require(args.year is None, \'Synthetic smoke has no validation year\')\n        # No package install, download, uv environment creation or CPU fallback.\n        for package in [\'torch\', \'numpy\', \'pandas\', \'scipy\', \'scikit-learn\', \'pyyaml\']:\n            report[\'environment\'][\'packages\'][package] = importlib.metadata.version(package)\n        uv = shutil.which(\'uv\')\n        require(uv is not None, \'uv is unavailable; no installation attempted. Use a reviewed hosted uv environment.\')\n        report[\'environment\'][\'uv\'] = subprocess.run([uv, \'--version\'], capture_output=True, text=True, check=True).stdout.strip()\n        config = Path(sys.prefix) / \'pyvenv.cfg\'\n        require(sys.prefix != sys.base_prefix and config.is_file()\n                and \'include-system-site-packages = true\' in config.read_text().lower(),\n                \'Run through the offline uv --system-site-packages environment from the notebook launcher\')\n        report[\'environment\'][\'venv\'] = {\'prefix\': sys.prefix, \'base_prefix\': sys.base_prefix,\n                                         \'system_site_packages\': True, \'package_actions\': \'none\'}\n        import torch\n        report[\'cuda\'] = cuda_environment(torch)\n        if args.mode == \'smoke\':\n            report[\'smoke\'] = synthetic_smoke(torch)\n        else:\n            smoke = json.loads(args.smoke_manifest.read_text())\n            require(smoke[\'status\'] == \'complete\' and smoke[\'mode\'] == \'smoke\'\n                    and smoke[\'protocol_sha256\'] == protocol_sha and smoke[\'smoke\'][\'synthetic_only\'] is True,\n                    \'A successful matching synthetic CUDA smoke is mandatory\')\n            require(smoke[\'cuda\'] == report[\'cuda\'], \'CUDA environment changed; rerun the synthetic smoke\')\n            report[\'smoke_manifest_sha256\'] = digest(args.smoke_manifest)\n            report[\'bundle\'] = stage_bundle(args.bundle, args.stage_root,\n                                             protocol[\'bundle_manifest_sha256\'], protocol[\'bundle_archive_sha256\'])\n            report[\'stage_root\'] = str(args.stage_root.resolve())\n            report[\'real_feature_table_loaded\'] = True\n            torch.cuda.reset_peak_memory_stats(0)\n            train_fold(torch, args.stage_root.resolve(), args.year, protocol, protocol_sha, out, report)\n        report[\'status\'] = \'complete\'\n    except BaseException:\n        report[\'status\'] = \'failed\'\n        report[\'error\'] = traceback.format_exc()\n        raise\n    finally:\n        shutil.copyfile(__file__, out / \'train_cuda_tcn.py\')\n        if args.protocol.is_file():\n            shutil.copyfile(args.protocol, out / \'protocol.json\')\n        report[\'runtime_seconds\'] = time.time() - started\n        report[\'artifacts\'] = {str(path.relative_to(out)): digest(path) for path in out.rglob(\'*\') if path.is_file() and path.name != \'manifest.json\'}\n        (out / \'manifest.json\').write_text(json.dumps(report, indent=2))\n        print(json.dumps({\'status\': report[\'status\'], \'mode\': args.mode, \'year\': args.year,\n                          \'manifest\': str(out / \'manifest.json\'), \'runtime_seconds\': report[\'runtime_seconds\']}), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'
PROTOCOL_TEXT = '{\n  "version": "cuda_tcn_v2_joint_anchor",\n  "status": "predeclared; hosted Torch/CUDA execution unverified",\n  "supersedes_protocol_sha256": "8c0aedf70538a546fdfbd586d8fa52783376e52d09a0386cdf39bc0bb4cd2c93",\n  "revision_reason": "Keep the original architecture/training design; move the NEW experiment anchor to the accepted joint blend. Do not modify the sealed input bundle or original MPS protocol.",\n  "trainer_sha256": "1afae39884da9d210a8ef695fb0c99636536bc688ce79961d502bf4c4b743e57",\n  "bundle_manifest_sha256": "6160e83f641dd3f311a1ebd96b2b3a05914b99b810e98ec262576be892563469",\n  "bundle_archive_sha256": "2dd110c37ae3bd81cda27104e09386a016bd88a8972e9c8708ee4518d410db44",\n  "feature_columns": [\n    "horizon_hour",\n    "horizon_day",\n    "hour_of_day",\n    "hour_sin",\n    "hour_cos",\n    "weekday",\n    "month",\n    "day_of_year",\n    "season_sin",\n    "season_cos",\n    "price_week_zscore",\n    "price_week_rank",\n    "volume_Kjelavatn",\n    "fill_Kjelavatn",\n    "price_over_water_Kjelavatn",\n    "headroom_after_inflow_Kjelavatn",\n    "volume_Staavatn",\n    "fill_Staavatn",\n    "price_over_water_Staavatn",\n    "headroom_after_inflow_Staavatn",\n    "volume_Langesae",\n    "fill_Langesae",\n    "price_over_water_Langesae",\n    "headroom_after_inflow_Langesae",\n    "volume_Foersvatn",\n    "fill_Foersvatn",\n    "price_over_water_Foersvatn",\n    "headroom_after_inflow_Foersvatn",\n    "volume_Venemo",\n    "fill_Venemo",\n    "price_over_water_Venemo",\n    "headroom_after_inflow_Venemo",\n    "volume_Botnedalsvatn",\n    "fill_Botnedalsvatn",\n    "price_over_water_Botnedalsvatn",\n    "headroom_after_inflow_Botnedalsvatn",\n    "volume_Bitdalsvatn",\n    "fill_Bitdalsvatn",\n    "price_over_water_Bitdalsvatn",\n    "headroom_after_inflow_Bitdalsvatn",\n    "volume_Vaamarvatn",\n    "fill_Vaamarvatn",\n    "price_over_water_Vaamarvatn",\n    "headroom_after_inflow_Vaamarvatn",\n    "volume_Langeidvatn",\n    "fill_Langeidvatn",\n    "price_over_water_Langeidvatn",\n    "headroom_after_inflow_Langeidvatn",\n    "volume_Totak",\n    "fill_Totak",\n    "price_over_water_Totak",\n    "headroom_after_inflow_Totak",\n    "volume_Songavatn",\n    "fill_Songavatn",\n    "price_over_water_Songavatn",\n    "headroom_after_inflow_Songavatn",\n    "volume_Vinjevatn",\n    "fill_Vinjevatn",\n    "price_over_water_Vinjevatn",\n    "headroom_after_inflow_Vinjevatn",\n    "volume_Bandak",\n    "fill_Bandak",\n    "price_over_water_Bandak",\n    "headroom_after_inflow_Bandak",\n    "volume_Bordalsvatn",\n    "fill_Bordalsvatn",\n    "price_over_water_Bordalsvatn",\n    "headroom_after_inflow_Bordalsvatn",\n    "volume_Byrtevatn",\n    "fill_Byrtevatn",\n    "price_over_water_Byrtevatn",\n    "headroom_after_inflow_Byrtevatn",\n    "volume_Hyljelihyl",\n    "fill_Hyljelihyl",\n    "price_over_water_Hyljelihyl",\n    "headroom_after_inflow_Hyljelihyl",\n    "volume_Vatjern",\n    "fill_Vatjern",\n    "price_over_water_Vatjern",\n    "headroom_after_inflow_Vatjern",\n    "inflow_Langesae",\n    "inflow_week_mean_Langesae",\n    "inflow_elapsed_mm3_Langesae",\n    "inflow_remaining_mm3_Langesae",\n    "inflow_Staavatn",\n    "inflow_week_mean_Staavatn",\n    "inflow_elapsed_mm3_Staavatn",\n    "inflow_remaining_mm3_Staavatn",\n    "inflow_Kjelavatn",\n    "inflow_week_mean_Kjelavatn",\n    "inflow_elapsed_mm3_Kjelavatn",\n    "inflow_remaining_mm3_Kjelavatn",\n    "inflow_Bordalsvatn",\n    "inflow_week_mean_Bordalsvatn",\n    "inflow_elapsed_mm3_Bordalsvatn",\n    "inflow_remaining_mm3_Bordalsvatn",\n    "inflow_Foersvatn",\n    "inflow_week_mean_Foersvatn",\n    "inflow_elapsed_mm3_Foersvatn",\n    "inflow_remaining_mm3_Foersvatn",\n    "inflow_Hyljelihyl",\n    "inflow_week_mean_Hyljelihyl",\n    "inflow_elapsed_mm3_Hyljelihyl",\n    "inflow_remaining_mm3_Hyljelihyl",\n    "inflow_Venemo",\n    "inflow_week_mean_Venemo",\n    "inflow_elapsed_mm3_Venemo",\n    "inflow_remaining_mm3_Venemo",\n    "inflow_Bitdalsvatn",\n    "inflow_week_mean_Bitdalsvatn",\n    "inflow_elapsed_mm3_Bitdalsvatn",\n    "inflow_remaining_mm3_Bitdalsvatn",\n    "inflow_Songavatn",\n    "inflow_week_mean_Songavatn",\n    "inflow_elapsed_mm3_Songavatn",\n    "inflow_remaining_mm3_Songavatn",\n    "inflow_Totak",\n    "inflow_week_mean_Totak",\n    "inflow_elapsed_mm3_Totak",\n    "inflow_remaining_mm3_Totak",\n    "inflow_Vaamarvatn",\n    "inflow_week_mean_Vaamarvatn",\n    "inflow_elapsed_mm3_Vaamarvatn",\n    "inflow_remaining_mm3_Vaamarvatn",\n    "inflow_Langeidvatn",\n    "inflow_week_mean_Langeidvatn",\n    "inflow_elapsed_mm3_Langeidvatn",\n    "inflow_remaining_mm3_Langeidvatn",\n    "inflow_Vatjern",\n    "inflow_week_mean_Vatjern",\n    "inflow_elapsed_mm3_Vatjern",\n    "inflow_remaining_mm3_Vatjern",\n    "inflow_Vinjevatn",\n    "inflow_week_mean_Vinjevatn",\n    "inflow_elapsed_mm3_Vinjevatn",\n    "inflow_remaining_mm3_Vinjevatn",\n    "inflow_Botnedalsvatn",\n    "inflow_week_mean_Botnedalsvatn",\n    "inflow_elapsed_mm3_Botnedalsvatn",\n    "inflow_remaining_mm3_Botnedalsvatn",\n    "inflow_Byrtevatn",\n    "inflow_week_mean_Byrtevatn",\n    "inflow_elapsed_mm3_Byrtevatn",\n    "inflow_remaining_mm3_Byrtevatn",\n    "inflow_Bandak",\n    "inflow_week_mean_Bandak",\n    "inflow_elapsed_mm3_Bandak",\n    "inflow_remaining_mm3_Bandak",\n    "inflow_r_Vest_Vassdraget",\n    "inflow_week_mean_r_Vest_Vassdraget",\n    "inflow_elapsed_mm3_r_Vest_Vassdraget",\n    "inflow_remaining_mm3_r_Vest_Vassdraget",\n    "min_volume_Totak",\n    "min_volume_week_max_Totak",\n    "min_volume_Byrtevatn",\n    "min_volume_week_max_Byrtevatn",\n    "min_volume_Staavatn",\n    "min_volume_week_max_Staavatn",\n    "min_flow_b_Kjelavatn",\n    "min_flow_week_max_b_Kjelavatn",\n    "min_flow_b_Hyljelihyl",\n    "min_flow_week_max_b_Hyljelihyl",\n    "min_flow_b_Vinjevatn",\n    "min_flow_week_max_b_Vinjevatn",\n    "min_flow_b_Byrtevatn",\n    "min_flow_week_max_b_Byrtevatn",\n    "min_flow_b_Bandak",\n    "min_flow_week_max_b_Bandak",\n    "min_flow_r_Vest_Vassdraget",\n    "min_flow_week_max_r_Vest_Vassdraget",\n    "physics_head_Hogga",\n    "physics_pool_fill_Hogga",\n    "physics_supply_fraction_Hogga",\n    "physics_catchment_inflow_fraction_Hogga",\n    "physics_dispatch_to_avoid_spill_Hogga",\n    "physics_profitable_remaining_fraction_Hogga",\n    "physics_head_Tokke",\n    "physics_pool_fill_Tokke",\n    "physics_supply_fraction_Tokke",\n    "physics_catchment_inflow_fraction_Tokke",\n    "physics_dispatch_to_avoid_spill_Tokke",\n    "physics_profitable_remaining_fraction_Tokke",\n    "physics_head_Lio",\n    "physics_pool_fill_Lio",\n    "physics_supply_fraction_Lio",\n    "physics_catchment_inflow_fraction_Lio",\n    "physics_dispatch_to_avoid_spill_Lio",\n    "physics_profitable_remaining_fraction_Lio",\n    "physics_head_Byrte",\n    "physics_pool_fill_Byrte",\n    "physics_supply_fraction_Byrte",\n    "physics_catchment_inflow_fraction_Byrte",\n    "physics_dispatch_to_avoid_spill_Byrte",\n    "physics_profitable_remaining_fraction_Byrte",\n    "physics_head_Vinje",\n    "physics_pool_fill_Vinje",\n    "physics_supply_fraction_Vinje",\n    "physics_catchment_inflow_fraction_Vinje",\n    "physics_dispatch_to_avoid_spill_Vinje",\n    "physics_profitable_remaining_fraction_Vinje",\n    "physics_head_Haukeli",\n    "physics_pool_fill_Haukeli",\n    "physics_supply_fraction_Haukeli",\n    "physics_catchment_inflow_fraction_Haukeli",\n    "physics_dispatch_to_avoid_spill_Haukeli",\n    "physics_profitable_remaining_fraction_Haukeli",\n    "physics_head_Songa",\n    "physics_pool_fill_Songa",\n    "physics_supply_fraction_Songa",\n    "physics_catchment_inflow_fraction_Songa",\n    "physics_dispatch_to_avoid_spill_Songa",\n    "physics_profitable_remaining_fraction_Songa",\n    "physics_head_Kjela",\n    "physics_pool_fill_Kjela",\n    "physics_supply_fraction_Kjela",\n    "physics_catchment_inflow_fraction_Kjela",\n    "physics_dispatch_to_avoid_spill_Kjela",\n    "physics_profitable_remaining_fraction_Kjela",\n    "physics_head_Vesle Kjela",\n    "physics_pool_fill_Vesle Kjela",\n    "physics_supply_fraction_Vesle Kjela",\n    "physics_catchment_inflow_fraction_Vesle Kjela",\n    "physics_dispatch_to_avoid_spill_Vesle Kjela",\n    "physics_profitable_remaining_fraction_Vesle Kjela",\n    "context_rank_remaining",\n    "lp_qmin_ratio_Hogga_G1",\n    "lp_power_fraction_Hogga_G1",\n    "lp_economic_margin_Hogga_G1",\n    "lp_qmin_ratio_Tokke_G1",\n    "lp_power_fraction_Tokke_G1",\n    "lp_economic_margin_Tokke_G1",\n    "lp_qmin_ratio_Tokke_G2",\n    "lp_power_fraction_Tokke_G2",\n    "lp_economic_margin_Tokke_G2",\n    "lp_qmin_ratio_Tokke_G3",\n    "lp_power_fraction_Tokke_G3",\n    "lp_economic_margin_Tokke_G3",\n    "lp_qmin_ratio_Tokke_G4",\n    "lp_power_fraction_Tokke_G4",\n    "lp_economic_margin_Tokke_G4",\n    "lp_qmin_ratio_Lio_G1",\n    "lp_power_fraction_Lio_G1",\n    "lp_economic_margin_Lio_G1",\n    "lp_qmin_ratio_Byrte_G1",\n    "lp_power_fraction_Byrte_G1",\n    "lp_economic_margin_Byrte_G1",\n    "lp_qmin_ratio_Vinje_G1",\n    "lp_power_fraction_Vinje_G1",\n    "lp_economic_margin_Vinje_G1",\n    "lp_qmin_ratio_Vinje_G2",\n    "lp_power_fraction_Vinje_G2",\n    "lp_economic_margin_Vinje_G2",\n    "lp_qmin_ratio_Vinje_G3",\n    "lp_power_fraction_Vinje_G3",\n    "lp_economic_margin_Vinje_G3",\n    "lp_qmin_ratio_Haukeli_G1",\n    "lp_power_fraction_Haukeli_G1",\n    "lp_economic_margin_Haukeli_G1",\n    "lp_qmin_ratio_Songa_G1",\n    "lp_power_fraction_Songa_G1",\n    "lp_economic_margin_Songa_G1",\n    "lp_qmin_ratio_Kjela_G1",\n    "lp_power_fraction_Kjela_G1",\n    "lp_economic_margin_Kjela_G1",\n    "lp_qmin_ratio_Vesle Kjela_G1",\n    "lp_power_fraction_Vesle Kjela_G1",\n    "lp_economic_margin_Vesle Kjela_G1",\n    "lp_plant_power_fraction_Hogga",\n    "lp_storage_fill_Hogga",\n    "lp_storage_min_slack_Hogga",\n    "lp_storage_max_slack_Hogga",\n    "lp_plant_power_fraction_Tokke",\n    "lp_storage_fill_Tokke",\n    "lp_storage_min_slack_Tokke",\n    "lp_storage_max_slack_Tokke",\n    "lp_plant_power_fraction_Lio",\n    "lp_storage_fill_Lio",\n    "lp_storage_min_slack_Lio",\n    "lp_storage_max_slack_Lio",\n    "lp_plant_power_fraction_Byrte",\n    "lp_storage_fill_Byrte",\n    "lp_storage_min_slack_Byrte",\n    "lp_storage_max_slack_Byrte",\n    "lp_plant_power_fraction_Vinje",\n    "lp_storage_fill_Vinje",\n    "lp_storage_min_slack_Vinje",\n    "lp_storage_max_slack_Vinje",\n    "lp_plant_power_fraction_Haukeli",\n    "lp_storage_fill_Haukeli",\n    "lp_storage_min_slack_Haukeli",\n    "lp_storage_max_slack_Haukeli",\n    "lp_plant_power_fraction_Songa",\n    "lp_storage_fill_Songa",\n    "lp_storage_min_slack_Songa",\n    "lp_storage_max_slack_Songa",\n    "lp_plant_power_fraction_Kjela",\n    "lp_storage_fill_Kjela",\n    "lp_storage_min_slack_Kjela",\n    "lp_storage_max_slack_Kjela",\n    "lp_plant_power_fraction_Vesle Kjela",\n    "lp_storage_fill_Vesle Kjela",\n    "lp_storage_min_slack_Vesle Kjela",\n    "lp_storage_max_slack_Vesle Kjela",\n    "norm_price",\n    "norm_price_offset_-24",\n    "norm_price_offset_-12",\n    "norm_price_offset_-6",\n    "norm_price_offset_-3",\n    "norm_price_offset_-1",\n    "norm_price_offset_+1",\n    "norm_price_offset_+3",\n    "norm_price_offset_+6",\n    "norm_price_offset_+12",\n    "norm_price_offset_+24",\n    "norm_price_week_mean",\n    "norm_price_week_std",\n    "norm_price_week_min",\n    "norm_price_week_max",\n    "norm_price_week_q10",\n    "norm_price_week_q25",\n    "norm_price_week_q50",\n    "norm_price_week_q75",\n    "norm_price_week_q90",\n    "norm_price_minus_week_mean",\n    "norm_price_day_mean",\n    "norm_price_day_min",\n    "norm_price_day_max",\n    "norm_price_day_std",\n    "norm_price_remaining_mean",\n    "norm_price_remaining_max",\n    "norm_water_value_Kjelavatn",\n    "norm_price_minus_water_Kjelavatn",\n    "norm_water_value_Staavatn",\n    "norm_price_minus_water_Staavatn",\n    "norm_water_value_Langesae",\n    "norm_price_minus_water_Langesae",\n    "norm_water_value_Foersvatn",\n    "norm_price_minus_water_Foersvatn",\n    "norm_water_value_Venemo",\n    "norm_price_minus_water_Venemo",\n    "norm_water_value_Botnedalsvatn",\n    "norm_price_minus_water_Botnedalsvatn",\n    "norm_water_value_Bitdalsvatn",\n    "norm_price_minus_water_Bitdalsvatn",\n    "norm_water_value_Vaamarvatn",\n    "norm_price_minus_water_Vaamarvatn",\n    "norm_water_value_Langeidvatn",\n    "norm_price_minus_water_Langeidvatn",\n    "norm_water_value_Totak",\n    "norm_price_minus_water_Totak",\n    "norm_water_value_Songavatn",\n    "norm_price_minus_water_Songavatn",\n    "norm_water_value_Vinjevatn",\n    "norm_price_minus_water_Vinjevatn",\n    "norm_water_value_Bandak",\n    "norm_price_minus_water_Bandak",\n    "norm_water_value_Bordalsvatn",\n    "norm_price_minus_water_Bordalsvatn",\n    "norm_water_value_Byrtevatn",\n    "norm_price_minus_water_Byrtevatn",\n    "norm_water_value_Hyljelihyl",\n    "norm_price_minus_water_Hyljelihyl",\n    "norm_water_value_Vatjern",\n    "norm_price_minus_water_Vatjern",\n    "norm_physics_switch_margin_minimum_Hogga",\n    "norm_physics_switch_margin_maximum_Hogga",\n    "norm_physics_switch_margin_minimum_Tokke",\n    "norm_physics_switch_margin_maximum_Tokke",\n    "norm_physics_switch_margin_minimum_Lio",\n    "norm_physics_switch_margin_maximum_Lio",\n    "norm_physics_switch_margin_minimum_Byrte",\n    "norm_physics_switch_margin_maximum_Byrte",\n    "norm_physics_switch_margin_minimum_Vinje",\n    "norm_physics_switch_margin_maximum_Vinje",\n    "norm_physics_switch_margin_minimum_Haukeli",\n    "norm_physics_switch_margin_maximum_Haukeli",\n    "norm_physics_switch_margin_minimum_Songa",\n    "norm_physics_switch_margin_maximum_Songa",\n    "norm_physics_switch_margin_minimum_Kjela",\n    "norm_physics_switch_margin_maximum_Kjela",\n    "norm_physics_switch_margin_minimum_Vesle Kjela",\n    "norm_physics_switch_margin_maximum_Vesle Kjela",\n    "norm_price_previous_mean_6h",\n    "norm_price_minus_previous_mean_6h",\n    "norm_price_next_mean_6h",\n    "norm_price_minus_next_mean_6h",\n    "norm_price_previous_mean_12h",\n    "norm_price_minus_previous_mean_12h",\n    "norm_price_next_mean_12h",\n    "norm_price_minus_next_mean_12h",\n    "norm_price_previous_mean_24h",\n    "norm_price_minus_previous_mean_24h",\n    "norm_price_next_mean_24h",\n    "norm_price_minus_next_mean_24h",\n    "monetary_scale_inverse"\n  ],\n  "feature_source": {\n    "scripts/analyze_predictions.py": "248ba174901e30765ebc2a1d2f1e24037734233950ba76a85ad0bf824099f8bf",\n    "scripts/build_lp_features.py": "8d2a4cf366b08ea08ba9c6b8483c1d89f20316272689846a53853e27e160f37d",\n    "scripts/case_calibration.py": "a93a3d7c007f21fea6beef262b3f9da7b1461157d9e8e11c109b70a4f271652c",\n    "scripts/economic_probe.py": "36629b6ebcf8952cafcf3e30ba90f6ee3347ab2ebd46de281631aa87fc9ab9c0",\n    "scripts/ensemble_compare.py": "668ebd1135922267e9d65e2451a934521f9205401f4b86d8813b0d3ff57b44a9",\n    "scripts/export_submission.py": "385c81cdc35a6a872463df605267f0bfa01eed55d8902a941db78747800bdb6d",\n    "scripts/plot_validation.py": "2e2945c3d49ef6e59045152c79c81d341436f5818711727d407ff891e1e62225",\n    "scripts/prefix_selection.py": "319f9a52762cc3a38c468b609b5557837b33164f1322a66834e05cebcc600193",\n    "scripts/repair_lp_cache_schema.py": "ab3a55b24b7739473390f3429a8018987275a1477a5b1e016dd29a48966327c5",\n    "scripts/score_lp_diagnostic.py": "8a823457b3e6e1f7d7274e9a861c017b45cc34a61a2115e25a231c6ddd3b6d4a",\n    "scripts/train.py": "f969ee755442fff4f47ccbaa8b592743f91e47ed5ecddb0c22a6b663d1c6cb23",\n    "sintef_project/context_features.py": "5a9d544dff941a9e810426cfe01c91fb9045fb8a4281a606dbf1b578ae01d3c0",\n    "sintef_project/features.py": "d6dd97719c125ca9af859c3251ca408ef05af541743ea2f1524d31961f2fba30",\n    "sintef_project/kaggle_metric.py": "27f9c7dcaff5a2a6be1d02f713a35d422825fad7c21ccec6f91471717ab828b6",\n    "sintef_project/lp_features.py": "bd16e0b4e9e32976b7eb50d5334e05b94f2b074bc0d8eb83a787f26f95268720",\n    "sintef_project/normalized.py": "2836de6aeffd992bf530abd2643df26650ade7c02e1c70e9aae1319bb5c3e16c",\n    "sintef_project/physics.py": "c9b0b2b81399a49c88802613a3ee85ba29fe97d5b7a3681f041d866972f82f41",\n    "sintef_project/postprocess.py": "ffbfa7974df4d0363f655bed00da1778aebe485988bdaef0309ac488f17c903b"\n  },\n  "data": {\n    "sintef_project/data/extended/Constraint_min_flow.csv": "146062702f58920440fc064d461d544e9cd7c503c8df74ce96e721bee04629ea",\n    "sintef_project/data/extended/Constraint_min_volume.csv": "8b639b8339b84c5ff7db34ba4b4618e99d05a03819bf99fa5032a54490278806",\n    "sintef_project/data/extended/Tokke_Vinje_topology.yaml": "29cfa233cec7fc4b5401d0d571a566e6e17b7bb7b46a175abcd37a53fe175497",\n    "sintef_project/data/kernel/Historical_day_ahead_price_2015_2025.csv": "e5edd141b34a61139f3ee3022305175446d4b4f74618df85552c4f2b5e556ee4",\n    "sintef_project/data/kernel/Historical_inflow_1958_2025.csv": "ff4f87d2768133b4a717c0ad366969244f7a86901695e1d4fef3c2ed8963ad66",\n    "sintef_project/data/kernel/Historical_volume_2015_2024.csv": "ef3d9c69699eef0d953b495ba69bf6676858e6b9c536087498135074912dd3b6",\n    "sintef_project/data/kernel/Synthetic_water_value_2015_2024.csv": "a5f09ff96b551d3c297d90ccf3a0bbe92d8814b537ad63a46579cb0ff44aa4cb",\n    "sintef_project/data/kernel/Unit_commitment_decisions.csv": "6d51f8cc6869cb82b44708168808c305c1e007cf79878ca1c1d316d07aae3ab3",\n    "sintef_project/prediction_mapping.csv": "c8b0e5bc500a5f1dec2337f5ac680847f6a7794a7c2b7a8fb19384910a029519",\n    "sintef_project/sample_submission.csv": "83e72b9e7299aff61065911af67437c81e8be2b0491ff012761d897c334f1cba"\n  },\n  "folds": {\n    "2021": {\n      "train_ids": [\n        1,\n        2,\n        3,\n        4,\n        5,\n        6,\n        7,\n        8,\n        9,\n        10,\n        11,\n        12,\n        13,\n        14,\n        15,\n        16,\n        17,\n        18,\n        19,\n        20,\n        21,\n        22,\n        23,\n        24,\n        25,\n        26,\n        27,\n        28,\n        29,\n        30,\n        31,\n        32,\n        33,\n        34,\n        35,\n        36,\n        37,\n        38,\n        39,\n        40,\n        41,\n        42,\n        43,\n        44,\n        45,\n        46,\n        47,\n        48,\n        49,\n        50,\n        51,\n        52,\n        53,\n        54,\n        55,\n        56,\n        57,\n        58,\n        59,\n        60,\n        61,\n        62,\n        63,\n        64,\n        65,\n        66,\n        67,\n        68,\n        69,\n        70,\n        71,\n        72,\n        73,\n        74,\n        75,\n        76,\n        77,\n        78,\n        79,\n        80,\n        81,\n        82,\n        83,\n        84,\n        85,\n        86,\n        87,\n        88,\n        89,\n        90,\n        91,\n        92,\n        93,\n        94,\n        95,\n        96,\n        97,\n        98,\n        99,\n        100,\n        101,\n        102,\n        103,\n        104,\n        105,\n        106,\n        107,\n        108,\n        109,\n        110,\n        111,\n        112,\n        113,\n        114,\n        115,\n        116,\n        117,\n        118,\n        119,\n        120,\n        121,\n        122,\n        123,\n        124,\n        125,\n        126,\n        127,\n        128,\n        129,\n        130,\n        131,\n        132,\n        133,\n        134,\n        135,\n        136,\n        137,\n        138,\n        139,\n        140,\n        141,\n        142,\n        143,\n        144,\n        145,\n        146,\n        147,\n        148,\n        149,\n        150,\n        151,\n        152,\n        153,\n        154,\n        155,\n        156,\n        157,\n        158,\n        159,\n        160,\n        161,\n        162,\n        163,\n        164,\n        165,\n        166,\n        167,\n        168,\n        169,\n        170,\n        171,\n        172,\n        173,\n        174,\n        175,\n        176,\n        177,\n        178,\n        179,\n        180,\n        181,\n        182,\n        183,\n        184,\n        185,\n        186,\n        187,\n        188,\n        189,\n        190,\n        191,\n        192,\n        193,\n        194,\n        195,\n        196,\n        197,\n        198,\n        199,\n        200,\n        201,\n        202,\n        203,\n        204,\n        205,\n        206,\n        207,\n        208,\n        209,\n        210,\n        211,\n        212,\n        213,\n        214,\n        215,\n        216,\n        217,\n        218,\n        219,\n        220,\n        221,\n        222,\n        223,\n        224,\n        225,\n        226,\n        227,\n        228,\n        229,\n        230,\n        231,\n        232,\n        233,\n        234,\n        235,\n        236,\n        237,\n        238,\n        239,\n        240,\n        241,\n        242,\n        243,\n        244,\n        245,\n        246,\n        247,\n        248,\n        249,\n        250,\n        251,\n        252,\n        253,\n        254,\n        255,\n        256,\n        257,\n        258,\n        259,\n        260,\n        261,\n        262,\n        263,\n        264,\n        265,\n        266,\n        267,\n        268,\n        269,\n        270,\n        271,\n        272,\n        273,\n        274,\n        275,\n        276,\n        277,\n        278,\n        279,\n        280,\n        281,\n        282,\n        283,\n        284,\n        285,\n        286,\n        287,\n        288,\n        289,\n        290,\n        291,\n        292,\n        293,\n        294,\n        295,\n        296,\n        297,\n        298,\n        299,\n        300,\n        301,\n        302,\n        303,\n        304,\n        305,\n        306,\n        307,\n        308,\n        309,\n        310,\n        311,\n        312,\n        313,\n        314,\n        315,\n        316,\n        317,\n        318,\n        319,\n        320,\n        321,\n        322,\n        323,\n        324,\n        325,\n        326,\n        327,\n        328,\n        329,\n        330,\n        331,\n        332,\n        333,\n        334,\n        335,\n        336,\n        337,\n        338,\n        339,\n        340,\n        341,\n        342,\n        343,\n        344,\n        345,\n        346,\n        347,\n        348,\n        349,\n        350,\n        351,\n        352,\n        353,\n        354,\n        355,\n        356,\n        357,\n        358,\n        359,\n        360,\n        361,\n        362,\n        363,\n        364,\n        365,\n        366,\n        367,\n        368,\n        369,\n        370,\n        371,\n        372,\n        373,\n        374,\n        375,\n        376,\n        377,\n        378,\n        379,\n        380,\n        381,\n        382,\n        383,\n        384,\n        385,\n        386,\n        387,\n        388,\n        389,\n        390,\n        391,\n        392,\n        393,\n        394,\n        395,\n        396,\n        397,\n        398,\n        399,\n        400,\n        401,\n        402,\n        403,\n        404,\n        405,\n        406,\n        407,\n        408,\n        409,\n        410,\n        411,\n        412,\n        413,\n        414,\n        415,\n        416,\n        417,\n        418,\n        419,\n        420,\n        421,\n        422,\n        423,\n        424,\n        425,\n        426,\n        427,\n        428,\n        429,\n        430,\n        431,\n        432,\n        433,\n        434,\n        435,\n        436,\n        437,\n        438,\n        439,\n        440,\n        441,\n        442,\n        443,\n        444,\n        445,\n        446,\n        447,\n        448,\n        449,\n        450,\n        451,\n        452,\n        453,\n        454,\n        455,\n        456,\n        457,\n        458,\n        459,\n        460,\n        461,\n        462,\n        463,\n        464,\n        465,\n        466,\n        467,\n        468,\n        469,\n        470,\n        471,\n        472,\n        473,\n        474,\n        475,\n        476,\n        477,\n        478,\n        479,\n        480,\n        481,\n        482,\n        483,\n        484,\n        485,\n        486,\n        487,\n        488,\n        489,\n        490,\n        491,\n        492,\n        493,\n        494,\n        495,\n        496,\n        497,\n        498,\n        499,\n        500,\n        501,\n        502,\n        503,\n        504,\n        505,\n        506,\n        507,\n        508,\n        509,\n        510,\n        511,\n        512,\n        513,\n        514,\n        515,\n        516,\n        517,\n        518,\n        519,\n        520,\n        521,\n        522,\n        523,\n        524,\n        525,\n        526,\n        527,\n        528,\n        529,\n        530,\n        531,\n        532,\n        533,\n        534,\n        535,\n        536,\n        537,\n        538,\n        539,\n        540,\n        541,\n        542,\n        543,\n        544,\n        545,\n        546,\n        547,\n        548,\n        549,\n        550,\n        551,\n        552,\n        553,\n        554,\n        555,\n        556,\n        557,\n        558,\n        559,\n        560,\n        561,\n        562,\n        563,\n        564,\n        565,\n        566,\n        567,\n        568,\n        569,\n        570,\n        571,\n        572,\n        573,\n        574,\n        575,\n        576,\n        577,\n        578,\n        579,\n        580,\n        581,\n        582,\n        583,\n        584,\n        585,\n        586,\n        587,\n        588,\n        589,\n        590,\n        591,\n        592,\n        593,\n        594,\n        595,\n        596,\n        597,\n        598,\n        599,\n        600,\n        601,\n        602,\n        603,\n        604,\n        605,\n        606,\n        607,\n        608,\n        609,\n        610,\n        611,\n        612,\n        613,\n        614,\n        615,\n        616,\n        617,\n        618,\n        619,\n        620,\n        621,\n        622,\n        623,\n        624,\n        625,\n        626,\n        627,\n        628,\n        629,\n        630,\n        631,\n        632,\n        633,\n        634,\n        635,\n        636,\n        637,\n        638,\n        639,\n        640,\n        641,\n        642,\n        643,\n        644,\n        645,\n        646,\n        647,\n        648,\n        649,\n        650,\n        651,\n        652,\n        653,\n        654,\n        655,\n        656,\n        657,\n        658,\n        659,\n        660,\n        661,\n        662,\n        663,\n        664,\n        665,\n        666,\n        667,\n        668,\n        669,\n        670,\n        671,\n        672,\n        673,\n        674,\n        675,\n        676,\n        677,\n        678,\n        679,\n        680,\n        681,\n        682,\n        683,\n        684,\n        685,\n        686,\n        687,\n        688,\n        689,\n        690,\n        691,\n        692,\n        693,\n        694,\n        695,\n        696,\n        697,\n        698,\n        699,\n        700,\n        701,\n        702,\n        703,\n        704,\n        705,\n        706,\n        707,\n        708,\n        709,\n        710,\n        711,\n        712,\n        713,\n        714,\n        715,\n        716,\n        717,\n        718,\n        719,\n        720,\n        721,\n        722,\n        723,\n        724,\n        725,\n        726,\n        727,\n        728,\n        729,\n        730,\n        731,\n        732,\n        733,\n        734,\n        735,\n        736,\n        737,\n        738,\n        739,\n        740,\n        741,\n        742,\n        743,\n        744,\n        745,\n        746,\n        747,\n        748,\n        749,\n        750,\n        751,\n        752,\n        753,\n        754,\n        755,\n        756,\n        757,\n        758,\n        759,\n        760,\n        761,\n        762,\n        763,\n        764,\n        765,\n        766,\n        767,\n        768,\n        769,\n        770,\n        771,\n        772,\n        773,\n        774,\n        775,\n        776,\n        777,\n        778,\n        779,\n        780,\n        781,\n        782,\n        783,\n        784,\n        785,\n        786,\n        787,\n        788,\n        789,\n        790,\n        791,\n        792,\n        793,\n        794,\n        795,\n        796,\n        797,\n        798,\n        799,\n        800,\n        801,\n        802,\n        803,\n        804,\n        805,\n        806,\n        807,\n        808,\n        809,\n        810,\n        811,\n        812,\n        813,\n        814,\n        815,\n        816,\n        817,\n        818,\n        819,\n        820,\n        821,\n        822,\n        823,\n        824,\n        825,\n        826,\n        827,\n        828,\n        829,\n        830,\n        831,\n        832,\n        833,\n        834,\n        835,\n        836,\n        837,\n        838,\n        839,\n        840,\n        841,\n        842,\n        843,\n        844,\n        845,\n        846,\n        847,\n        848,\n        849,\n        850,\n        851,\n        852,\n        853,\n        854,\n        855,\n        856,\n        857,\n        858,\n        859,\n        860,\n        861,\n        862,\n        863,\n        864,\n        865,\n        866,\n        867,\n        868,\n        869,\n        870,\n        871,\n        872,\n        873,\n        874,\n        875,\n        876,\n        877,\n        878,\n        879,\n        880,\n        881,\n        882,\n        883,\n        884,\n        885,\n        886,\n        887,\n        888,\n        889,\n        890,\n        891,\n        892,\n        893,\n        894,\n        895,\n        896,\n        897,\n        898,\n        899,\n        900,\n        901,\n        902,\n        903,\n        904,\n        905,\n        906,\n        907,\n        908,\n        909,\n        910,\n        911,\n        912,\n        913,\n        914,\n        915,\n        916,\n        917,\n        918,\n        919,\n        920,\n        921,\n        922,\n        923,\n        924,\n        925,\n        926,\n        927,\n        928,\n        929,\n        930,\n        931,\n        932,\n        933,\n        934,\n        935,\n        936,\n        937,\n        938,\n        939,\n        940,\n        941,\n        942,\n        943,\n        944,\n        945,\n        946,\n        947,\n        948,\n        949,\n        950,\n        951,\n        952,\n        953,\n        954,\n        955,\n        956,\n        957,\n        958,\n        959,\n        960,\n        961,\n        962,\n        963,\n        964,\n        965,\n        966,\n        967,\n        968,\n        969,\n        970,\n        971,\n        972,\n        973,\n        974,\n        975,\n        976,\n        977,\n        978,\n        979,\n        980,\n        981,\n        982,\n        983,\n        984,\n        985,\n        986,\n        987,\n        988,\n        989,\n        990,\n        991,\n        992,\n        993,\n        994,\n        995,\n        996,\n        997,\n        998,\n        999,\n        1000,\n        1001,\n        1002,\n        1003,\n        1004,\n        1005,\n        1006,\n        1007,\n        1008,\n        1009,\n        1010,\n        1011,\n        1012,\n        1013,\n        1014,\n        1015,\n        1016,\n        1017,\n        1018,\n        1019,\n        1020,\n        1021,\n        1022,\n        1023,\n        1024,\n        1025,\n        1026,\n        1027,\n        1028,\n        1029,\n        1030,\n        1031,\n        1032,\n        1033,\n        1034,\n        1035,\n        1036,\n        1037,\n        1038,\n        1039,\n        1040,\n        1041,\n        1042,\n        1043,\n        1044,\n        1045,\n        1046,\n        1047,\n        1048,\n        1049,\n        1050,\n        1051,\n        1052,\n        1053,\n        1054,\n        1055,\n        1056,\n        1057,\n        1058,\n        1059,\n        1060,\n        1061,\n        1062,\n        1063,\n        1064,\n        1065,\n        1066,\n        1067,\n        1068,\n        1069,\n        1070,\n        1071,\n        1072,\n        1073,\n        1074,\n        1075,\n        1076,\n        1077,\n        1078,\n        1079,\n        1080,\n        1081,\n        1082,\n        1083,\n        1084,\n        1085,\n        1086,\n        1087,\n        1088,\n        1089,\n        1090,\n        1091,\n        1092,\n        1093,\n        1094,\n        1095,\n        1096,\n        1097,\n        1098,\n        1099,\n        1100,\n        1101,\n        1102,\n        1103,\n        1104,\n        1105,\n        1106,\n        1107,\n        1108,\n        1109,\n        1110,\n        1111,\n        1112,\n        1113,\n        1114,\n        1115,\n        1116,\n        1117,\n        1118,\n        1119,\n        1120,\n        1121,\n        1122,\n        1123,\n        1124,\n        1125,\n        1126,\n        1127,\n        1128,\n        1129,\n        1130,\n        1131,\n        1132,\n        1133,\n        1134,\n        1135,\n        1136,\n        1137,\n        1138,\n        1139,\n        1140,\n        1141,\n        1142,\n        1143,\n        1144,\n        1145,\n        1146,\n        1147,\n        1148,\n        1149,\n        1150,\n        1151,\n        1152,\n        1153,\n        1154,\n        1155,\n        1156,\n        1157,\n        1158,\n        1159,\n        1160,\n        1161,\n        1162,\n        1163,\n        1164,\n        1165,\n        1166,\n        1167,\n        1168,\n        1169,\n        1170,\n        1171,\n        1172,\n        1173,\n        1174,\n        1175,\n        1176,\n        1177,\n        1178,\n        1179,\n        1180,\n        1181,\n        1182,\n        1183,\n        1184,\n        1185,\n        1186,\n        1187,\n        1188,\n        1189,\n        1190,\n        1191,\n        1192,\n        1193,\n        1194,\n        1195,\n        1196,\n        1197,\n        1198,\n        1199,\n        1200,\n        1201,\n        1202,\n        1203,\n        1204,\n        1205,\n        1206,\n        1207,\n        1208,\n        1209,\n        1210,\n        1211,\n        1212,\n        1213,\n        1214,\n        1215,\n        1216,\n        1217,\n        1218,\n        1219,\n        1220,\n        1221,\n        1222,\n        1223,\n        1224,\n        1225,\n        1226,\n        1227,\n        1228,\n        1229,\n        1230,\n        1231,\n        1232,\n        1233,\n        1234,\n        1235,\n        1236,\n        1237,\n        1238,\n        1239,\n        1240,\n        1241,\n        1242,\n        1243,\n        1244,\n        1245,\n        1246,\n        1247,\n        1248,\n        1249,\n        1250,\n        1251,\n        1252,\n        1253,\n        1254,\n        1255,\n        1256,\n        1257,\n        1258,\n        1259,\n        1260,\n        1261,\n        1262,\n        1263,\n        1264,\n        1265,\n        1266,\n        1267,\n        1268,\n        1269,\n        1270,\n        1271,\n        1272,\n        1273,\n        1274,\n        1275,\n        1276,\n        1277,\n        1278,\n        1279,\n        1280,\n        1281,\n        1282,\n        1283,\n        1284,\n        1285,\n        1286,\n        1287,\n        1288,\n        1289,\n        1290,\n        1291,\n        1292,\n        1293,\n        1294,\n        1295,\n        1296,\n        1297,\n        1298,\n        1299,\n        1300,\n        1301,\n        1302,\n        1303,\n        1304,\n        1305,\n        1306,\n        1307,\n        1308,\n        1309,\n        1310,\n        1311,\n        1312,\n        1313,\n        1314,\n        1315,\n        1316,\n        1317,\n        1318,\n        1319,\n        1320,\n        1321,\n        1322,\n        1323,\n        1324,\n        1325,\n        1326,\n        1327,\n        1328,\n        1329,\n        1330,\n        1331,\n        1332,\n        1333,\n        1334,\n        1335,\n        1336,\n        1337,\n        1338,\n        1339,\n        1340,\n        1341,\n        1342,\n        1343,\n        1344,\n        1345,\n        1346,\n        1347,\n        1348,\n        1349,\n        1350,\n        1351,\n        1352,\n        1353,\n        1354,\n        1355,\n        1356,\n        1357,\n        1358,\n        1359,\n        1360,\n        1361,\n        1362,\n        1363,\n        1364,\n        1365,\n        1366,\n        1367,\n        1368,\n        1369,\n        1370,\n        1371,\n        1372,\n        1373,\n        1374,\n        1375,\n        1376,\n        1377,\n        1378,\n        1379,\n        1380,\n        1381,\n        1382,\n        1383,\n        1384,\n        1385,\n        1386,\n        1387,\n        1388,\n        1389,\n        1390,\n        1391,\n        1392,\n        1393,\n        1394,\n        1395,\n        1396,\n        1397,\n        1398,\n        1399,\n        1400,\n        1401,\n        1402,\n        1403,\n        1404,\n        1405,\n        1406,\n        1407,\n        1408,\n        1409,\n        1410,\n        1411,\n        1412,\n        1413,\n        1414,\n        1415,\n        1416,\n        1417,\n        1418,\n        1419,\n        1420,\n        1421,\n        1422,\n        1423,\n        1424,\n        1425,\n        1426,\n        1427,\n        1428,\n        1429,\n        1430,\n        1431,\n        1432,\n        1433,\n        1434,\n        1435,\n        1436,\n        1437,\n        1438,\n        1439,\n        1440,\n        1441,\n        1442,\n        1443,\n        1444,\n        1445,\n        1446,\n        1447,\n        1448,\n        1449,\n        1450,\n        1451,\n        1452,\n        1453,\n        1454,\n        1455,\n        1456,\n        1457,\n        1458,\n        1459,\n        1460,\n        1461,\n        1462,\n        1463,\n        1464,\n        1465,\n        1466,\n        1467,\n        1468,\n        1469,\n        1470,\n        1471,\n        1472,\n        1473,\n        1474,\n        1475,\n        1476,\n        1477,\n        1478,\n        1479,\n        1480,\n        1481,\n        1482,\n        1483,\n        1484,\n        1485,\n        1486,\n        1487,\n        1488,\n        1489,\n        1490,\n        1491,\n        1492,\n        1493,\n        1494,\n        1495,\n        1496,\n        1497,\n        1498,\n        1499,\n        1500,\n        1501,\n        1502,\n        1503,\n        1504,\n        1505,\n        1506,\n        1507,\n        1508,\n        1509,\n        1510,\n        1511,\n        1512,\n        1513,\n        1514,\n        1515,\n        1516,\n        1517,\n        1518,\n        1519,\n        1520,\n        1521,\n        1522,\n        1523,\n        1524,\n        1525,\n        1526,\n        1527,\n        1528,\n        1529,\n        1530,\n        1531,\n        1532,\n        1533,\n        1534,\n        1535,\n        1536,\n        1537,\n        1538,\n        1539,\n        1540,\n        1541,\n        1542,\n        1543,\n        1544,\n        1545,\n        1546,\n        1547,\n        1548,\n        1549,\n        1550,\n        1551,\n        1552,\n        1553,\n        1554,\n        1555,\n        1556,\n        1557,\n        1558,\n        1559,\n        1560,\n        1561,\n        1562,\n        1563,\n        1564,\n        1565,\n        1566,\n        1567,\n        1568,\n        1569,\n        1570,\n        1571,\n        1572,\n        1573,\n        1574,\n        1575,\n        1576,\n        1577,\n        1578,\n        1579,\n        1580,\n        1581,\n        1582,\n        1583,\n        1584,\n        1585,\n        1586,\n        1587,\n        1588,\n        1589,\n        1590,\n        1591,\n        1592,\n        1593,\n        1594,\n        1595,\n        1596,\n        1597,\n        1598,\n        1599,\n        1600,\n        1601,\n        1602,\n        1603,\n        1604,\n        1605,\n        1606,\n        1607,\n        1608,\n        1609,\n        1610,\n        1611,\n        1612,\n        1613,\n        1614,\n        1615,\n        1616,\n        1617,\n        1618,\n        1619,\n        1620,\n        1621,\n        1622,\n        1623,\n        1624,\n        1625,\n        1626,\n        1627,\n        1628,\n        1629,\n        1630,\n        1631,\n        1632,\n        1633,\n        1634,\n        1635,\n        1636,\n        1637,\n        1638,\n        1639,\n        1640,\n        1641,\n        1642,\n        1643,\n        1644,\n        1645,\n        1646,\n        1647,\n        1648,\n        1649,\n        1650,\n        1651,\n        1652,\n        1653,\n        1654,\n        1655,\n        1656,\n        1657,\n        1658,\n        1659,\n        1660,\n        1661,\n        1662,\n        1663,\n        1664,\n        1665,\n        1666,\n        1667,\n        1668,\n        1669,\n        1670,\n        1671,\n        1672,\n        1673,\n        1674,\n        1675,\n        1676,\n        1677,\n        1678,\n        1679,\n        1680,\n        1681,\n        1682,\n        1683,\n        1684,\n        1685,\n        1686,\n        1687,\n        1688,\n        1689,\n        1690,\n        1691,\n        1692,\n        1693,\n        1694,\n        1695,\n        1696,\n        1697,\n        1698,\n        1699,\n        1700,\n        1701,\n        1702,\n        1703,\n        1704,\n        1705,\n        1706,\n        1707,\n        1708,\n        1709,\n        1710,\n        1711,\n        1712,\n        1713,\n        1714,\n        1715,\n        1716,\n        1717,\n        1718,\n        1719,\n        1720,\n        1721,\n        1722,\n        1723,\n        1724,\n        1725,\n        1726,\n        1727,\n        1728,\n        1729,\n        1730,\n        1731,\n        1732,\n        1733,\n        1734,\n        1735,\n        1736,\n        1737,\n        1738,\n        1739,\n        1740,\n        1741,\n        1742,\n        1743,\n        1744,\n        1745,\n        1746,\n        1747,\n        1748,\n        1749,\n        1750,\n        1751,\n        1752,\n        1753,\n        1754,\n        1755,\n        1756,\n        1757,\n        1758,\n        1759,\n        1760,\n        1761,\n        1762,\n        1763,\n        1764,\n        1765,\n        1766,\n        1767,\n        1768,\n        1769,\n        1770,\n        1771,\n        1772,\n        1773,\n        1774,\n        1775,\n        1776,\n        1777,\n        1778,\n        1779,\n        1780,\n        1781,\n        1782,\n        1783,\n        1784,\n        1785,\n        1786,\n        1787,\n        1788,\n        1789,\n        1790,\n        1791,\n        1792,\n        1793,\n        1794,\n        1795,\n        1796,\n        1797,\n        1798,\n        1799,\n        1800,\n        1801,\n        1802,\n        1803,\n        1804,\n        1805,\n        1806,\n        1807,\n        1808,\n        1809,\n        1810,\n        1811,\n        1812,\n        1813,\n        1814,\n        1815,\n        1816,\n        1817,\n        1818,\n        1819,\n        1820,\n        1821,\n        1822,\n        1823,\n        1824,\n        1825,\n        1826,\n        1827,\n        1828,\n        1829,\n        1830,\n        1831,\n        1832,\n        1833,\n        1834,\n        1835,\n        1836,\n        1837,\n        1838,\n        1839,\n        1840,\n        1841,\n        1842,\n        1843,\n        1844,\n        1845,\n        1846,\n        1847,\n        1848,\n        1849,\n        1850,\n        1851,\n        1852,\n        1853,\n        1854,\n        1855,\n        1856,\n        1857,\n        1858,\n        1859,\n        1860,\n        1861,\n        1862,\n        1863,\n        1864,\n        1865,\n        1866,\n        1867,\n        1868,\n        1869,\n        1870,\n        1871,\n        1872,\n        1873,\n        1874,\n        1875,\n        1876,\n        1877,\n        1878,\n        1879,\n        1880,\n        1881,\n        1882,\n        1883,\n        1884,\n        1885,\n        1886,\n        1887,\n        1888,\n        1889,\n        1890,\n        1891,\n        1892,\n        1893,\n        1894,\n        1895,\n        1896,\n        1897,\n        1898,\n        1899,\n        1900,\n        1901,\n        1902,\n        1903,\n        1904,\n        1905,\n        1906,\n        1907,\n        1908,\n        1909,\n        1910,\n        1911,\n        1912,\n        1913,\n        1914,\n        1915,\n        1916,\n        1917,\n        1918,\n        1919,\n        1920,\n        1921,\n        1922,\n        1923,\n        1924,\n        1925,\n        1926,\n        1927,\n        1928,\n        1929,\n        1930,\n        1931,\n        1932,\n        1933,\n        1934,\n        1935,\n        1936,\n        1937,\n        1938,\n        1939,\n        1940,\n        1941,\n        1942,\n        1943,\n        1944,\n        1945,\n        1946,\n        1947,\n        1948,\n        1949,\n        1950,\n        1951,\n        1952,\n        1953,\n        1954,\n        1955,\n        1956,\n        1957,\n        1958,\n        1959,\n        1960,\n        1961,\n        1962,\n        1963,\n        1964,\n        1965,\n        1966,\n        1967,\n        1968,\n        1969,\n        1970,\n        1971,\n        1972,\n        1973,\n        1974,\n        1975,\n        1976,\n        1977,\n        1978,\n        1979,\n        1980,\n        1981,\n        1982,\n        1983,\n        1984,\n        1985,\n        1986,\n        1987,\n        1988,\n        1989,\n        1990,\n        1991,\n        1992,\n        1993,\n        1994,\n        1995,\n        1996,\n        1997,\n        1998,\n        1999,\n        2000,\n        2001,\n        2002,\n        2003,\n        2004,\n        2005,\n        2006,\n        2007,\n        2008,\n        2009,\n        2010,\n        2011,\n        2012,\n        2013,\n        2014,\n        2015,\n        2016,\n        2017,\n        2018,\n        2019,\n        2020,\n        2021,\n        2022,\n        2023,\n        2024,\n        2025,\n        2026,\n        2027,\n        2028,\n        2029,\n        2030,\n        2031,\n        2032,\n        2033,\n        2034,\n        2035,\n        2036,\n        2037,\n        2038,\n        2039,\n        2040,\n        2041,\n        2042,\n        2043,\n        2044,\n        2045,\n        2046,\n        2047,\n        2048,\n        2049,\n        2050,\n        2051,\n        2052,\n        2053,\n        2054,\n        2055,\n        2056,\n        2057,\n        2058,\n        2059,\n        2060,\n        2061,\n        2062,\n        2063,\n        2064,\n        2065,\n        2066,\n        2067,\n        2068,\n        2069,\n        2070,\n        2071,\n        2072,\n        2073,\n        2074,\n        2075,\n        2076,\n        2077,\n        2078,\n        2079,\n        2080,\n        2081,\n        2082,\n        2083,\n        2084,\n        2085,\n        2086,\n        2087,\n        2088,\n        2089,\n        2090,\n        2091,\n        2092,\n        2093,\n        2094,\n        2095,\n        2096,\n        2097,\n        2098,\n        2099,\n        2100,\n        2101,\n        2102,\n        2103,\n        2104,\n        2105,\n        2106,\n        2107,\n        2108,\n        2109,\n        2110,\n        2111,\n        2112,\n        2113,\n        2114,\n        2115,\n        2116,\n        2117,\n        2118,\n        2119,\n        2120,\n        2121,\n        2122,\n        2123,\n        2124,\n        2125,\n        2126,\n        2127,\n        2128,\n        2129,\n        2130,\n        2131,\n        2132,\n        2133,\n        2134,\n        2135,\n        2136,\n        2137,\n        2138,\n        2139,\n        2140,\n        2141,\n        2142,\n        2143,\n        2144,\n        2145,\n        2146,\n        2147,\n        2148,\n        2149,\n        2150,\n        2151,\n        2152,\n        2153,\n        2154,\n        2155,\n        2156,\n        2157,\n        2158,\n        2159,\n        2160,\n        2161,\n        2162,\n        2163,\n        2164,\n        2165,\n        2166,\n        2167,\n        2168,\n        2169,\n        2170,\n        2171,\n        2172,\n        2173,\n        2174,\n        2175,\n        2176,\n        2177,\n        2178,\n        2179,\n        2180,\n        2181,\n        2182,\n        2183,\n        2184,\n        2185,\n        2186\n      ],\n      "validation_ids": [\n        2193,\n        2194,\n        2195,\n        2196,\n        2197,\n        2198,\n        2199,\n        2200,\n        2201,\n        2202,\n        2203,\n        2204,\n        2205,\n        2206,\n        2207,\n        2208,\n        2209,\n        2210,\n        2211,\n        2212,\n        2213,\n        2214,\n        2215,\n        2216,\n        2217,\n        2218,\n        2219,\n        2220,\n        2221,\n        2222,\n        2223,\n        2224,\n        2225,\n        2226,\n        2227,\n        2228,\n        2229,\n        2230,\n        2231,\n        2232,\n        2233,\n        2234,\n        2235,\n        2236,\n        2237,\n        2238,\n        2239,\n        2240,\n        2241,\n        2242,\n        2243,\n        2244,\n        2245,\n        2246,\n        2247,\n        2248,\n        2249,\n        2250,\n        2251,\n        2252,\n        2253,\n        2254,\n        2255,\n        2256,\n        2257,\n        2258,\n        2259,\n        2260,\n        2261,\n        2262,\n        2263,\n        2264,\n        2265,\n        2266,\n        2267,\n        2268,\n        2269,\n        2270,\n        2271,\n        2272,\n        2273,\n        2274,\n        2275,\n        2276,\n        2277,\n        2278,\n        2279,\n        2280,\n        2281,\n        2282,\n        2283,\n        2284,\n        2285,\n        2286,\n        2287,\n        2288,\n        2289,\n        2290,\n        2291,\n        2292,\n        2293,\n        2294,\n        2295,\n        2296,\n        2297,\n        2298,\n        2299,\n        2300,\n        2301,\n        2302,\n        2303,\n        2304,\n        2305,\n        2306,\n        2307,\n        2308,\n        2309,\n        2310,\n        2311,\n        2312,\n        2313,\n        2314,\n        2315,\n        2316,\n        2317,\n        2318,\n        2319,\n        2320,\n        2321,\n        2322,\n        2323,\n        2324,\n        2325,\n        2326,\n        2327,\n        2328,\n        2329,\n        2330,\n        2331,\n        2332,\n        2333,\n        2334,\n        2335,\n        2336,\n        2337,\n        2338,\n        2339,\n        2340,\n        2341,\n        2342,\n        2343,\n        2344,\n        2345,\n        2346,\n        2347,\n        2348,\n        2349,\n        2350,\n        2351,\n        2352,\n        2353,\n        2354,\n        2355,\n        2356,\n        2357,\n        2358,\n        2359,\n        2360,\n        2361,\n        2362,\n        2363,\n        2364,\n        2365,\n        2366,\n        2367,\n        2368,\n        2369,\n        2370,\n        2371,\n        2372,\n        2373,\n        2374,\n        2375,\n        2376,\n        2377,\n        2378,\n        2379,\n        2380,\n        2381,\n        2382,\n        2383,\n        2384,\n        2385,\n        2386,\n        2387,\n        2388,\n        2389,\n        2390,\n        2391,\n        2392,\n        2393,\n        2394,\n        2395,\n        2396,\n        2397,\n        2398,\n        2399,\n        2400,\n        2401,\n        2402,\n        2403,\n        2404,\n        2405,\n        2406,\n        2407,\n        2408,\n        2409,\n        2410,\n        2411,\n        2412,\n        2413,\n        2414,\n        2415,\n        2416,\n        2417,\n        2418,\n        2419,\n        2420,\n        2421,\n        2422,\n        2423,\n        2424,\n        2425,\n        2426,\n        2427,\n        2428,\n        2429,\n        2430,\n        2431,\n        2432,\n        2433,\n        2434,\n        2435,\n        2436,\n        2437,\n        2438,\n        2439,\n        2440,\n        2441,\n        2442,\n        2443,\n        2444,\n        2445,\n        2446,\n        2447,\n        2448,\n        2449,\n        2450,\n        2451,\n        2452,\n        2453,\n        2454,\n        2455,\n        2456,\n        2457,\n        2458,\n        2459,\n        2460,\n        2461,\n        2462,\n        2463,\n        2464,\n        2465,\n        2466,\n        2467,\n        2468,\n        2469,\n        2470,\n        2471,\n        2472,\n        2473,\n        2474,\n        2475,\n        2476,\n        2477,\n        2478,\n        2479,\n        2480,\n        2481,\n        2482,\n        2483,\n        2484,\n        2485,\n        2486,\n        2487,\n        2488,\n        2489,\n        2490,\n        2491,\n        2492,\n        2493,\n        2494,\n        2495,\n        2496,\n        2497,\n        2498,\n        2499,\n        2500,\n        2501,\n        2502,\n        2503,\n        2504,\n        2505,\n        2506,\n        2507,\n        2508,\n        2509,\n        2510,\n        2511,\n        2512,\n        2513,\n        2514,\n        2515,\n        2516,\n        2517,\n        2518,\n        2519,\n        2520,\n        2521,\n        2522,\n        2523,\n        2524,\n        2525,\n        2526,\n        2527,\n        2528,\n        2529,\n        2530,\n        2531,\n        2532,\n        2533,\n        2534,\n        2535,\n        2536,\n        2537,\n        2538,\n        2539,\n        2540,\n        2541,\n        2542,\n        2543,\n        2544,\n        2545,\n        2546,\n        2547,\n        2548,\n        2549,\n        2550,\n        2551,\n        2552,\n        2553,\n        2554,\n        2555,\n        2556,\n        2557\n      ],\n      "train_dates": [\n        "2015-01-01 00:00:00+00:00",\n        "2020-12-25 00:00:00+00:00"\n      ],\n      "validation_dates": [\n        "2021-01-01 00:00:00+00:00",\n        "2021-12-31 00:00:00+00:00"\n      ],\n      "purge_days": 6\n    },\n    "2022": {\n      "train_ids": [\n        1,\n        2,\n        3,\n        4,\n        5,\n        6,\n        7,\n        8,\n        9,\n        10,\n        11,\n        12,\n        13,\n        14,\n        15,\n        16,\n        17,\n        18,\n        19,\n        20,\n        21,\n        22,\n        23,\n        24,\n        25,\n        26,\n        27,\n        28,\n        29,\n        30,\n        31,\n        32,\n        33,\n        34,\n        35,\n        36,\n        37,\n        38,\n        39,\n        40,\n        41,\n        42,\n        43,\n        44,\n        45,\n        46,\n        47,\n        48,\n        49,\n        50,\n        51,\n        52,\n        53,\n        54,\n        55,\n        56,\n        57,\n        58,\n        59,\n        60,\n        61,\n        62,\n        63,\n        64,\n        65,\n        66,\n        67,\n        68,\n        69,\n        70,\n        71,\n        72,\n        73,\n        74,\n        75,\n        76,\n        77,\n        78,\n        79,\n        80,\n        81,\n        82,\n        83,\n        84,\n        85,\n        86,\n        87,\n        88,\n        89,\n        90,\n        91,\n        92,\n        93,\n        94,\n        95,\n        96,\n        97,\n        98,\n        99,\n        100,\n        101,\n        102,\n        103,\n        104,\n        105,\n        106,\n        107,\n        108,\n        109,\n        110,\n        111,\n        112,\n        113,\n        114,\n        115,\n        116,\n        117,\n        118,\n        119,\n        120,\n        121,\n        122,\n        123,\n        124,\n        125,\n        126,\n        127,\n        128,\n        129,\n        130,\n        131,\n        132,\n        133,\n        134,\n        135,\n        136,\n        137,\n        138,\n        139,\n        140,\n        141,\n        142,\n        143,\n        144,\n        145,\n        146,\n        147,\n        148,\n        149,\n        150,\n        151,\n        152,\n        153,\n        154,\n        155,\n        156,\n        157,\n        158,\n        159,\n        160,\n        161,\n        162,\n        163,\n        164,\n        165,\n        166,\n        167,\n        168,\n        169,\n        170,\n        171,\n        172,\n        173,\n        174,\n        175,\n        176,\n        177,\n        178,\n        179,\n        180,\n        181,\n        182,\n        183,\n        184,\n        185,\n        186,\n        187,\n        188,\n        189,\n        190,\n        191,\n        192,\n        193,\n        194,\n        195,\n        196,\n        197,\n        198,\n        199,\n        200,\n        201,\n        202,\n        203,\n        204,\n        205,\n        206,\n        207,\n        208,\n        209,\n        210,\n        211,\n        212,\n        213,\n        214,\n        215,\n        216,\n        217,\n        218,\n        219,\n        220,\n        221,\n        222,\n        223,\n        224,\n        225,\n        226,\n        227,\n        228,\n        229,\n        230,\n        231,\n        232,\n        233,\n        234,\n        235,\n        236,\n        237,\n        238,\n        239,\n        240,\n        241,\n        242,\n        243,\n        244,\n        245,\n        246,\n        247,\n        248,\n        249,\n        250,\n        251,\n        252,\n        253,\n        254,\n        255,\n        256,\n        257,\n        258,\n        259,\n        260,\n        261,\n        262,\n        263,\n        264,\n        265,\n        266,\n        267,\n        268,\n        269,\n        270,\n        271,\n        272,\n        273,\n        274,\n        275,\n        276,\n        277,\n        278,\n        279,\n        280,\n        281,\n        282,\n        283,\n        284,\n        285,\n        286,\n        287,\n        288,\n        289,\n        290,\n        291,\n        292,\n        293,\n        294,\n        295,\n        296,\n        297,\n        298,\n        299,\n        300,\n        301,\n        302,\n        303,\n        304,\n        305,\n        306,\n        307,\n        308,\n        309,\n        310,\n        311,\n        312,\n        313,\n        314,\n        315,\n        316,\n        317,\n        318,\n        319,\n        320,\n        321,\n        322,\n        323,\n        324,\n        325,\n        326,\n        327,\n        328,\n        329,\n        330,\n        331,\n        332,\n        333,\n        334,\n        335,\n        336,\n        337,\n        338,\n        339,\n        340,\n        341,\n        342,\n        343,\n        344,\n        345,\n        346,\n        347,\n        348,\n        349,\n        350,\n        351,\n        352,\n        353,\n        354,\n        355,\n        356,\n        357,\n        358,\n        359,\n        360,\n        361,\n        362,\n        363,\n        364,\n        365,\n        366,\n        367,\n        368,\n        369,\n        370,\n        371,\n        372,\n        373,\n        374,\n        375,\n        376,\n        377,\n        378,\n        379,\n        380,\n        381,\n        382,\n        383,\n        384,\n        385,\n        386,\n        387,\n        388,\n        389,\n        390,\n        391,\n        392,\n        393,\n        394,\n        395,\n        396,\n        397,\n        398,\n        399,\n        400,\n        401,\n        402,\n        403,\n        404,\n        405,\n        406,\n        407,\n        408,\n        409,\n        410,\n        411,\n        412,\n        413,\n        414,\n        415,\n        416,\n        417,\n        418,\n        419,\n        420,\n        421,\n        422,\n        423,\n        424,\n        425,\n        426,\n        427,\n        428,\n        429,\n        430,\n        431,\n        432,\n        433,\n        434,\n        435,\n        436,\n        437,\n        438,\n        439,\n        440,\n        441,\n        442,\n        443,\n        444,\n        445,\n        446,\n        447,\n        448,\n        449,\n        450,\n        451,\n        452,\n        453,\n        454,\n        455,\n        456,\n        457,\n        458,\n        459,\n        460,\n        461,\n        462,\n        463,\n        464,\n        465,\n        466,\n        467,\n        468,\n        469,\n        470,\n        471,\n        472,\n        473,\n        474,\n        475,\n        476,\n        477,\n        478,\n        479,\n        480,\n        481,\n        482,\n        483,\n        484,\n        485,\n        486,\n        487,\n        488,\n        489,\n        490,\n        491,\n        492,\n        493,\n        494,\n        495,\n        496,\n        497,\n        498,\n        499,\n        500,\n        501,\n        502,\n        503,\n        504,\n        505,\n        506,\n        507,\n        508,\n        509,\n        510,\n        511,\n        512,\n        513,\n        514,\n        515,\n        516,\n        517,\n        518,\n        519,\n        520,\n        521,\n        522,\n        523,\n        524,\n        525,\n        526,\n        527,\n        528,\n        529,\n        530,\n        531,\n        532,\n        533,\n        534,\n        535,\n        536,\n        537,\n        538,\n        539,\n        540,\n        541,\n        542,\n        543,\n        544,\n        545,\n        546,\n        547,\n        548,\n        549,\n        550,\n        551,\n        552,\n        553,\n        554,\n        555,\n        556,\n        557,\n        558,\n        559,\n        560,\n        561,\n        562,\n        563,\n        564,\n        565,\n        566,\n        567,\n        568,\n        569,\n        570,\n        571,\n        572,\n        573,\n        574,\n        575,\n        576,\n        577,\n        578,\n        579,\n        580,\n        581,\n        582,\n        583,\n        584,\n        585,\n        586,\n        587,\n        588,\n        589,\n        590,\n        591,\n        592,\n        593,\n        594,\n        595,\n        596,\n        597,\n        598,\n        599,\n        600,\n        601,\n        602,\n        603,\n        604,\n        605,\n        606,\n        607,\n        608,\n        609,\n        610,\n        611,\n        612,\n        613,\n        614,\n        615,\n        616,\n        617,\n        618,\n        619,\n        620,\n        621,\n        622,\n        623,\n        624,\n        625,\n        626,\n        627,\n        628,\n        629,\n        630,\n        631,\n        632,\n        633,\n        634,\n        635,\n        636,\n        637,\n        638,\n        639,\n        640,\n        641,\n        642,\n        643,\n        644,\n        645,\n        646,\n        647,\n        648,\n        649,\n        650,\n        651,\n        652,\n        653,\n        654,\n        655,\n        656,\n        657,\n        658,\n        659,\n        660,\n        661,\n        662,\n        663,\n        664,\n        665,\n        666,\n        667,\n        668,\n        669,\n        670,\n        671,\n        672,\n        673,\n        674,\n        675,\n        676,\n        677,\n        678,\n        679,\n        680,\n        681,\n        682,\n        683,\n        684,\n        685,\n        686,\n        687,\n        688,\n        689,\n        690,\n        691,\n        692,\n        693,\n        694,\n        695,\n        696,\n        697,\n        698,\n        699,\n        700,\n        701,\n        702,\n        703,\n        704,\n        705,\n        706,\n        707,\n        708,\n        709,\n        710,\n        711,\n        712,\n        713,\n        714,\n        715,\n        716,\n        717,\n        718,\n        719,\n        720,\n        721,\n        722,\n        723,\n        724,\n        725,\n        726,\n        727,\n        728,\n        729,\n        730,\n        731,\n        732,\n        733,\n        734,\n        735,\n        736,\n        737,\n        738,\n        739,\n        740,\n        741,\n        742,\n        743,\n        744,\n        745,\n        746,\n        747,\n        748,\n        749,\n        750,\n        751,\n        752,\n        753,\n        754,\n        755,\n        756,\n        757,\n        758,\n        759,\n        760,\n        761,\n        762,\n        763,\n        764,\n        765,\n        766,\n        767,\n        768,\n        769,\n        770,\n        771,\n        772,\n        773,\n        774,\n        775,\n        776,\n        777,\n        778,\n        779,\n        780,\n        781,\n        782,\n        783,\n        784,\n        785,\n        786,\n        787,\n        788,\n        789,\n        790,\n        791,\n        792,\n        793,\n        794,\n        795,\n        796,\n        797,\n        798,\n        799,\n        800,\n        801,\n        802,\n        803,\n        804,\n        805,\n        806,\n        807,\n        808,\n        809,\n        810,\n        811,\n        812,\n        813,\n        814,\n        815,\n        816,\n        817,\n        818,\n        819,\n        820,\n        821,\n        822,\n        823,\n        824,\n        825,\n        826,\n        827,\n        828,\n        829,\n        830,\n        831,\n        832,\n        833,\n        834,\n        835,\n        836,\n        837,\n        838,\n        839,\n        840,\n        841,\n        842,\n        843,\n        844,\n        845,\n        846,\n        847,\n        848,\n        849,\n        850,\n        851,\n        852,\n        853,\n        854,\n        855,\n        856,\n        857,\n        858,\n        859,\n        860,\n        861,\n        862,\n        863,\n        864,\n        865,\n        866,\n        867,\n        868,\n        869,\n        870,\n        871,\n        872,\n        873,\n        874,\n        875,\n        876,\n        877,\n        878,\n        879,\n        880,\n        881,\n        882,\n        883,\n        884,\n        885,\n        886,\n        887,\n        888,\n        889,\n        890,\n        891,\n        892,\n        893,\n        894,\n        895,\n        896,\n        897,\n        898,\n        899,\n        900,\n        901,\n        902,\n        903,\n        904,\n        905,\n        906,\n        907,\n        908,\n        909,\n        910,\n        911,\n        912,\n        913,\n        914,\n        915,\n        916,\n        917,\n        918,\n        919,\n        920,\n        921,\n        922,\n        923,\n        924,\n        925,\n        926,\n        927,\n        928,\n        929,\n        930,\n        931,\n        932,\n        933,\n        934,\n        935,\n        936,\n        937,\n        938,\n        939,\n        940,\n        941,\n        942,\n        943,\n        944,\n        945,\n        946,\n        947,\n        948,\n        949,\n        950,\n        951,\n        952,\n        953,\n        954,\n        955,\n        956,\n        957,\n        958,\n        959,\n        960,\n        961,\n        962,\n        963,\n        964,\n        965,\n        966,\n        967,\n        968,\n        969,\n        970,\n        971,\n        972,\n        973,\n        974,\n        975,\n        976,\n        977,\n        978,\n        979,\n        980,\n        981,\n        982,\n        983,\n        984,\n        985,\n        986,\n        987,\n        988,\n        989,\n        990,\n        991,\n        992,\n        993,\n        994,\n        995,\n        996,\n        997,\n        998,\n        999,\n        1000,\n        1001,\n        1002,\n        1003,\n        1004,\n        1005,\n        1006,\n        1007,\n        1008,\n        1009,\n        1010,\n        1011,\n        1012,\n        1013,\n        1014,\n        1015,\n        1016,\n        1017,\n        1018,\n        1019,\n        1020,\n        1021,\n        1022,\n        1023,\n        1024,\n        1025,\n        1026,\n        1027,\n        1028,\n        1029,\n        1030,\n        1031,\n        1032,\n        1033,\n        1034,\n        1035,\n        1036,\n        1037,\n        1038,\n        1039,\n        1040,\n        1041,\n        1042,\n        1043,\n        1044,\n        1045,\n        1046,\n        1047,\n        1048,\n        1049,\n        1050,\n        1051,\n        1052,\n        1053,\n        1054,\n        1055,\n        1056,\n        1057,\n        1058,\n        1059,\n        1060,\n        1061,\n        1062,\n        1063,\n        1064,\n        1065,\n        1066,\n        1067,\n        1068,\n        1069,\n        1070,\n        1071,\n        1072,\n        1073,\n        1074,\n        1075,\n        1076,\n        1077,\n        1078,\n        1079,\n        1080,\n        1081,\n        1082,\n        1083,\n        1084,\n        1085,\n        1086,\n        1087,\n        1088,\n        1089,\n        1090,\n        1091,\n        1092,\n        1093,\n        1094,\n        1095,\n        1096,\n        1097,\n        1098,\n        1099,\n        1100,\n        1101,\n        1102,\n        1103,\n        1104,\n        1105,\n        1106,\n        1107,\n        1108,\n        1109,\n        1110,\n        1111,\n        1112,\n        1113,\n        1114,\n        1115,\n        1116,\n        1117,\n        1118,\n        1119,\n        1120,\n        1121,\n        1122,\n        1123,\n        1124,\n        1125,\n        1126,\n        1127,\n        1128,\n        1129,\n        1130,\n        1131,\n        1132,\n        1133,\n        1134,\n        1135,\n        1136,\n        1137,\n        1138,\n        1139,\n        1140,\n        1141,\n        1142,\n        1143,\n        1144,\n        1145,\n        1146,\n        1147,\n        1148,\n        1149,\n        1150,\n        1151,\n        1152,\n        1153,\n        1154,\n        1155,\n        1156,\n        1157,\n        1158,\n        1159,\n        1160,\n        1161,\n        1162,\n        1163,\n        1164,\n        1165,\n        1166,\n        1167,\n        1168,\n        1169,\n        1170,\n        1171,\n        1172,\n        1173,\n        1174,\n        1175,\n        1176,\n        1177,\n        1178,\n        1179,\n        1180,\n        1181,\n        1182,\n        1183,\n        1184,\n        1185,\n        1186,\n        1187,\n        1188,\n        1189,\n        1190,\n        1191,\n        1192,\n        1193,\n        1194,\n        1195,\n        1196,\n        1197,\n        1198,\n        1199,\n        1200,\n        1201,\n        1202,\n        1203,\n        1204,\n        1205,\n        1206,\n        1207,\n        1208,\n        1209,\n        1210,\n        1211,\n        1212,\n        1213,\n        1214,\n        1215,\n        1216,\n        1217,\n        1218,\n        1219,\n        1220,\n        1221,\n        1222,\n        1223,\n        1224,\n        1225,\n        1226,\n        1227,\n        1228,\n        1229,\n        1230,\n        1231,\n        1232,\n        1233,\n        1234,\n        1235,\n        1236,\n        1237,\n        1238,\n        1239,\n        1240,\n        1241,\n        1242,\n        1243,\n        1244,\n        1245,\n        1246,\n        1247,\n        1248,\n        1249,\n        1250,\n        1251,\n        1252,\n        1253,\n        1254,\n        1255,\n        1256,\n        1257,\n        1258,\n        1259,\n        1260,\n        1261,\n        1262,\n        1263,\n        1264,\n        1265,\n        1266,\n        1267,\n        1268,\n        1269,\n        1270,\n        1271,\n        1272,\n        1273,\n        1274,\n        1275,\n        1276,\n        1277,\n        1278,\n        1279,\n        1280,\n        1281,\n        1282,\n        1283,\n        1284,\n        1285,\n        1286,\n        1287,\n        1288,\n        1289,\n        1290,\n        1291,\n        1292,\n        1293,\n        1294,\n        1295,\n        1296,\n        1297,\n        1298,\n        1299,\n        1300,\n        1301,\n        1302,\n        1303,\n        1304,\n        1305,\n        1306,\n        1307,\n        1308,\n        1309,\n        1310,\n        1311,\n        1312,\n        1313,\n        1314,\n        1315,\n        1316,\n        1317,\n        1318,\n        1319,\n        1320,\n        1321,\n        1322,\n        1323,\n        1324,\n        1325,\n        1326,\n        1327,\n        1328,\n        1329,\n        1330,\n        1331,\n        1332,\n        1333,\n        1334,\n        1335,\n        1336,\n        1337,\n        1338,\n        1339,\n        1340,\n        1341,\n        1342,\n        1343,\n        1344,\n        1345,\n        1346,\n        1347,\n        1348,\n        1349,\n        1350,\n        1351,\n        1352,\n        1353,\n        1354,\n        1355,\n        1356,\n        1357,\n        1358,\n        1359,\n        1360,\n        1361,\n        1362,\n        1363,\n        1364,\n        1365,\n        1366,\n        1367,\n        1368,\n        1369,\n        1370,\n        1371,\n        1372,\n        1373,\n        1374,\n        1375,\n        1376,\n        1377,\n        1378,\n        1379,\n        1380,\n        1381,\n        1382,\n        1383,\n        1384,\n        1385,\n        1386,\n        1387,\n        1388,\n        1389,\n        1390,\n        1391,\n        1392,\n        1393,\n        1394,\n        1395,\n        1396,\n        1397,\n        1398,\n        1399,\n        1400,\n        1401,\n        1402,\n        1403,\n        1404,\n        1405,\n        1406,\n        1407,\n        1408,\n        1409,\n        1410,\n        1411,\n        1412,\n        1413,\n        1414,\n        1415,\n        1416,\n        1417,\n        1418,\n        1419,\n        1420,\n        1421,\n        1422,\n        1423,\n        1424,\n        1425,\n        1426,\n        1427,\n        1428,\n        1429,\n        1430,\n        1431,\n        1432,\n        1433,\n        1434,\n        1435,\n        1436,\n        1437,\n        1438,\n        1439,\n        1440,\n        1441,\n        1442,\n        1443,\n        1444,\n        1445,\n        1446,\n        1447,\n        1448,\n        1449,\n        1450,\n        1451,\n        1452,\n        1453,\n        1454,\n        1455,\n        1456,\n        1457,\n        1458,\n        1459,\n        1460,\n        1461,\n        1462,\n        1463,\n        1464,\n        1465,\n        1466,\n        1467,\n        1468,\n        1469,\n        1470,\n        1471,\n        1472,\n        1473,\n        1474,\n        1475,\n        1476,\n        1477,\n        1478,\n        1479,\n        1480,\n        1481,\n        1482,\n        1483,\n        1484,\n        1485,\n        1486,\n        1487,\n        1488,\n        1489,\n        1490,\n        1491,\n        1492,\n        1493,\n        1494,\n        1495,\n        1496,\n        1497,\n        1498,\n        1499,\n        1500,\n        1501,\n        1502,\n        1503,\n        1504,\n        1505,\n        1506,\n        1507,\n        1508,\n        1509,\n        1510,\n        1511,\n        1512,\n        1513,\n        1514,\n        1515,\n        1516,\n        1517,\n        1518,\n        1519,\n        1520,\n        1521,\n        1522,\n        1523,\n        1524,\n        1525,\n        1526,\n        1527,\n        1528,\n        1529,\n        1530,\n        1531,\n        1532,\n        1533,\n        1534,\n        1535,\n        1536,\n        1537,\n        1538,\n        1539,\n        1540,\n        1541,\n        1542,\n        1543,\n        1544,\n        1545,\n        1546,\n        1547,\n        1548,\n        1549,\n        1550,\n        1551,\n        1552,\n        1553,\n        1554,\n        1555,\n        1556,\n        1557,\n        1558,\n        1559,\n        1560,\n        1561,\n        1562,\n        1563,\n        1564,\n        1565,\n        1566,\n        1567,\n        1568,\n        1569,\n        1570,\n        1571,\n        1572,\n        1573,\n        1574,\n        1575,\n        1576,\n        1577,\n        1578,\n        1579,\n        1580,\n        1581,\n        1582,\n        1583,\n        1584,\n        1585,\n        1586,\n        1587,\n        1588,\n        1589,\n        1590,\n        1591,\n        1592,\n        1593,\n        1594,\n        1595,\n        1596,\n        1597,\n        1598,\n        1599,\n        1600,\n        1601,\n        1602,\n        1603,\n        1604,\n        1605,\n        1606,\n        1607,\n        1608,\n        1609,\n        1610,\n        1611,\n        1612,\n        1613,\n        1614,\n        1615,\n        1616,\n        1617,\n        1618,\n        1619,\n        1620,\n        1621,\n        1622,\n        1623,\n        1624,\n        1625,\n        1626,\n        1627,\n        1628,\n        1629,\n        1630,\n        1631,\n        1632,\n        1633,\n        1634,\n        1635,\n        1636,\n        1637,\n        1638,\n        1639,\n        1640,\n        1641,\n        1642,\n        1643,\n        1644,\n        1645,\n        1646,\n        1647,\n        1648,\n        1649,\n        1650,\n        1651,\n        1652,\n        1653,\n        1654,\n        1655,\n        1656,\n        1657,\n        1658,\n        1659,\n        1660,\n        1661,\n        1662,\n        1663,\n        1664,\n        1665,\n        1666,\n        1667,\n        1668,\n        1669,\n        1670,\n        1671,\n        1672,\n        1673,\n        1674,\n        1675,\n        1676,\n        1677,\n        1678,\n        1679,\n        1680,\n        1681,\n        1682,\n        1683,\n        1684,\n        1685,\n        1686,\n        1687,\n        1688,\n        1689,\n        1690,\n        1691,\n        1692,\n        1693,\n        1694,\n        1695,\n        1696,\n        1697,\n        1698,\n        1699,\n        1700,\n        1701,\n        1702,\n        1703,\n        1704,\n        1705,\n        1706,\n        1707,\n        1708,\n        1709,\n        1710,\n        1711,\n        1712,\n        1713,\n        1714,\n        1715,\n        1716,\n        1717,\n        1718,\n        1719,\n        1720,\n        1721,\n        1722,\n        1723,\n        1724,\n        1725,\n        1726,\n        1727,\n        1728,\n        1729,\n        1730,\n        1731,\n        1732,\n        1733,\n        1734,\n        1735,\n        1736,\n        1737,\n        1738,\n        1739,\n        1740,\n        1741,\n        1742,\n        1743,\n        1744,\n        1745,\n        1746,\n        1747,\n        1748,\n        1749,\n        1750,\n        1751,\n        1752,\n        1753,\n        1754,\n        1755,\n        1756,\n        1757,\n        1758,\n        1759,\n        1760,\n        1761,\n        1762,\n        1763,\n        1764,\n        1765,\n        1766,\n        1767,\n        1768,\n        1769,\n        1770,\n        1771,\n        1772,\n        1773,\n        1774,\n        1775,\n        1776,\n        1777,\n        1778,\n        1779,\n        1780,\n        1781,\n        1782,\n        1783,\n        1784,\n        1785,\n        1786,\n        1787,\n        1788,\n        1789,\n        1790,\n        1791,\n        1792,\n        1793,\n        1794,\n        1795,\n        1796,\n        1797,\n        1798,\n        1799,\n        1800,\n        1801,\n        1802,\n        1803,\n        1804,\n        1805,\n        1806,\n        1807,\n        1808,\n        1809,\n        1810,\n        1811,\n        1812,\n        1813,\n        1814,\n        1815,\n        1816,\n        1817,\n        1818,\n        1819,\n        1820,\n        1821,\n        1822,\n        1823,\n        1824,\n        1825,\n        1826,\n        1827,\n        1828,\n        1829,\n        1830,\n        1831,\n        1832,\n        1833,\n        1834,\n        1835,\n        1836,\n        1837,\n        1838,\n        1839,\n        1840,\n        1841,\n        1842,\n        1843,\n        1844,\n        1845,\n        1846,\n        1847,\n        1848,\n        1849,\n        1850,\n        1851,\n        1852,\n        1853,\n        1854,\n        1855,\n        1856,\n        1857,\n        1858,\n        1859,\n        1860,\n        1861,\n        1862,\n        1863,\n        1864,\n        1865,\n        1866,\n        1867,\n        1868,\n        1869,\n        1870,\n        1871,\n        1872,\n        1873,\n        1874,\n        1875,\n        1876,\n        1877,\n        1878,\n        1879,\n        1880,\n        1881,\n        1882,\n        1883,\n        1884,\n        1885,\n        1886,\n        1887,\n        1888,\n        1889,\n        1890,\n        1891,\n        1892,\n        1893,\n        1894,\n        1895,\n        1896,\n        1897,\n        1898,\n        1899,\n        1900,\n        1901,\n        1902,\n        1903,\n        1904,\n        1905,\n        1906,\n        1907,\n        1908,\n        1909,\n        1910,\n        1911,\n        1912,\n        1913,\n        1914,\n        1915,\n        1916,\n        1917,\n        1918,\n        1919,\n        1920,\n        1921,\n        1922,\n        1923,\n        1924,\n        1925,\n        1926,\n        1927,\n        1928,\n        1929,\n        1930,\n        1931,\n        1932,\n        1933,\n        1934,\n        1935,\n        1936,\n        1937,\n        1938,\n        1939,\n        1940,\n        1941,\n        1942,\n        1943,\n        1944,\n        1945,\n        1946,\n        1947,\n        1948,\n        1949,\n        1950,\n        1951,\n        1952,\n        1953,\n        1954,\n        1955,\n        1956,\n        1957,\n        1958,\n        1959,\n        1960,\n        1961,\n        1962,\n        1963,\n        1964,\n        1965,\n        1966,\n        1967,\n        1968,\n        1969,\n        1970,\n        1971,\n        1972,\n        1973,\n        1974,\n        1975,\n        1976,\n        1977,\n        1978,\n        1979,\n        1980,\n        1981,\n        1982,\n        1983,\n        1984,\n        1985,\n        1986,\n        1987,\n        1988,\n        1989,\n        1990,\n        1991,\n        1992,\n        1993,\n        1994,\n        1995,\n        1996,\n        1997,\n        1998,\n        1999,\n        2000,\n        2001,\n        2002,\n        2003,\n        2004,\n        2005,\n        2006,\n        2007,\n        2008,\n        2009,\n        2010,\n        2011,\n        2012,\n        2013,\n        2014,\n        2015,\n        2016,\n        2017,\n        2018,\n        2019,\n        2020,\n        2021,\n        2022,\n        2023,\n        2024,\n        2025,\n        2026,\n        2027,\n        2028,\n        2029,\n        2030,\n        2031,\n        2032,\n        2033,\n        2034,\n        2035,\n        2036,\n        2037,\n        2038,\n        2039,\n        2040,\n        2041,\n        2042,\n        2043,\n        2044,\n        2045,\n        2046,\n        2047,\n        2048,\n        2049,\n        2050,\n        2051,\n        2052,\n        2053,\n        2054,\n        2055,\n        2056,\n        2057,\n        2058,\n        2059,\n        2060,\n        2061,\n        2062,\n        2063,\n        2064,\n        2065,\n        2066,\n        2067,\n        2068,\n        2069,\n        2070,\n        2071,\n        2072,\n        2073,\n        2074,\n        2075,\n        2076,\n        2077,\n        2078,\n        2079,\n        2080,\n        2081,\n        2082,\n        2083,\n        2084,\n        2085,\n        2086,\n        2087,\n        2088,\n        2089,\n        2090,\n        2091,\n        2092,\n        2093,\n        2094,\n        2095,\n        2096,\n        2097,\n        2098,\n        2099,\n        2100,\n        2101,\n        2102,\n        2103,\n        2104,\n        2105,\n        2106,\n        2107,\n        2108,\n        2109,\n        2110,\n        2111,\n        2112,\n        2113,\n        2114,\n        2115,\n        2116,\n        2117,\n        2118,\n        2119,\n        2120,\n        2121,\n        2122,\n        2123,\n        2124,\n        2125,\n        2126,\n        2127,\n        2128,\n        2129,\n        2130,\n        2131,\n        2132,\n        2133,\n        2134,\n        2135,\n        2136,\n        2137,\n        2138,\n        2139,\n        2140,\n        2141,\n        2142,\n        2143,\n        2144,\n        2145,\n        2146,\n        2147,\n        2148,\n        2149,\n        2150,\n        2151,\n        2152,\n        2153,\n        2154,\n        2155,\n        2156,\n        2157,\n        2158,\n        2159,\n        2160,\n        2161,\n        2162,\n        2163,\n        2164,\n        2165,\n        2166,\n        2167,\n        2168,\n        2169,\n        2170,\n        2171,\n        2172,\n        2173,\n        2174,\n        2175,\n        2176,\n        2177,\n        2178,\n        2179,\n        2180,\n        2181,\n        2182,\n        2183,\n        2184,\n        2185,\n        2186,\n        2187,\n        2188,\n        2189,\n        2190,\n        2191,\n        2192,\n        2193,\n        2194,\n        2195,\n        2196,\n        2197,\n        2198,\n        2199,\n        2200,\n        2201,\n        2202,\n        2203,\n        2204,\n        2205,\n        2206,\n        2207,\n        2208,\n        2209,\n        2210,\n        2211,\n        2212,\n        2213,\n        2214,\n        2215,\n        2216,\n        2217,\n        2218,\n        2219,\n        2220,\n        2221,\n        2222,\n        2223,\n        2224,\n        2225,\n        2226,\n        2227,\n        2228,\n        2229,\n        2230,\n        2231,\n        2232,\n        2233,\n        2234,\n        2235,\n        2236,\n        2237,\n        2238,\n        2239,\n        2240,\n        2241,\n        2242,\n        2243,\n        2244,\n        2245,\n        2246,\n        2247,\n        2248,\n        2249,\n        2250,\n        2251,\n        2252,\n        2253,\n        2254,\n        2255,\n        2256,\n        2257,\n        2258,\n        2259,\n        2260,\n        2261,\n        2262,\n        2263,\n        2264,\n        2265,\n        2266,\n        2267,\n        2268,\n        2269,\n        2270,\n        2271,\n        2272,\n        2273,\n        2274,\n        2275,\n        2276,\n        2277,\n        2278,\n        2279,\n        2280,\n        2281,\n        2282,\n        2283,\n        2284,\n        2285,\n        2286,\n        2287,\n        2288,\n        2289,\n        2290,\n        2291,\n        2292,\n        2293,\n        2294,\n        2295,\n        2296,\n        2297,\n        2298,\n        2299,\n        2300,\n        2301,\n        2302,\n        2303,\n        2304,\n        2305,\n        2306,\n        2307,\n        2308,\n        2309,\n        2310,\n        2311,\n        2312,\n        2313,\n        2314,\n        2315,\n        2316,\n        2317,\n        2318,\n        2319,\n        2320,\n        2321,\n        2322,\n        2323,\n        2324,\n        2325,\n        2326,\n        2327,\n        2328,\n        2329,\n        2330,\n        2331,\n        2332,\n        2333,\n        2334,\n        2335,\n        2336,\n        2337,\n        2338,\n        2339,\n        2340,\n        2341,\n        2342,\n        2343,\n        2344,\n        2345,\n        2346,\n        2347,\n        2348,\n        2349,\n        2350,\n        2351,\n        2352,\n        2353,\n        2354,\n        2355,\n        2356,\n        2357,\n        2358,\n        2359,\n        2360,\n        2361,\n        2362,\n        2363,\n        2364,\n        2365,\n        2366,\n        2367,\n        2368,\n        2369,\n        2370,\n        2371,\n        2372,\n        2373,\n        2374,\n        2375,\n        2376,\n        2377,\n        2378,\n        2379,\n        2380,\n        2381,\n        2382,\n        2383,\n        2384,\n        2385,\n        2386,\n        2387,\n        2388,\n        2389,\n        2390,\n        2391,\n        2392,\n        2393,\n        2394,\n        2395,\n        2396,\n        2397,\n        2398,\n        2399,\n        2400,\n        2401,\n        2402,\n        2403,\n        2404,\n        2405,\n        2406,\n        2407,\n        2408,\n        2409,\n        2410,\n        2411,\n        2412,\n        2413,\n        2414,\n        2415,\n        2416,\n        2417,\n        2418,\n        2419,\n        2420,\n        2421,\n        2422,\n        2423,\n        2424,\n        2425,\n        2426,\n        2427,\n        2428,\n        2429,\n        2430,\n        2431,\n        2432,\n        2433,\n        2434,\n        2435,\n        2436,\n        2437,\n        2438,\n        2439,\n        2440,\n        2441,\n        2442,\n        2443,\n        2444,\n        2445,\n        2446,\n        2447,\n        2448,\n        2449,\n        2450,\n        2451,\n        2452,\n        2453,\n        2454,\n        2455,\n        2456,\n        2457,\n        2458,\n        2459,\n        2460,\n        2461,\n        2462,\n        2463,\n        2464,\n        2465,\n        2466,\n        2467,\n        2468,\n        2469,\n        2470,\n        2471,\n        2472,\n        2473,\n        2474,\n        2475,\n        2476,\n        2477,\n        2478,\n        2479,\n        2480,\n        2481,\n        2482,\n        2483,\n        2484,\n        2485,\n        2486,\n        2487,\n        2488,\n        2489,\n        2490,\n        2491,\n        2492,\n        2493,\n        2494,\n        2495,\n        2496,\n        2497,\n        2498,\n        2499,\n        2500,\n        2501,\n        2502,\n        2503,\n        2504,\n        2505,\n        2506,\n        2507,\n        2508,\n        2509,\n        2510,\n        2511,\n        2512,\n        2513,\n        2514,\n        2515,\n        2516,\n        2517,\n        2518,\n        2519,\n        2520,\n        2521,\n        2522,\n        2523,\n        2524,\n        2525,\n        2526,\n        2527,\n        2528,\n        2529,\n        2530,\n        2531,\n        2532,\n        2533,\n        2534,\n        2535,\n        2536,\n        2537,\n        2538,\n        2539,\n        2540,\n        2541,\n        2542,\n        2543,\n        2544,\n        2545,\n        2546,\n        2547,\n        2548,\n        2549,\n        2550,\n        2551\n      ],\n      "validation_ids": [\n        2558,\n        2559,\n        2560,\n        2561,\n        2562,\n        2563,\n        2564,\n        2565,\n        2566,\n        2567,\n        2568,\n        2569,\n        2570,\n        2571,\n        2572,\n        2573,\n        2574,\n        2575,\n        2576,\n        2577,\n        2578,\n        2579,\n        2580,\n        2581,\n        2582,\n        2583,\n        2584,\n        2585,\n        2586,\n        2587,\n        2588,\n        2589,\n        2590,\n        2591,\n        2592,\n        2593,\n        2594,\n        2595,\n        2596,\n        2597,\n        2598,\n        2599,\n        2600,\n        2601,\n        2602,\n        2603,\n        2604,\n        2605,\n        2606,\n        2607,\n        2608,\n        2609,\n        2610,\n        2611,\n        2612,\n        2613,\n        2614,\n        2615,\n        2616,\n        2617,\n        2618,\n        2619,\n        2620,\n        2621,\n        2622,\n        2623,\n        2624,\n        2625,\n        2626,\n        2627,\n        2628,\n        2629,\n        2630,\n        2631,\n        2632,\n        2633,\n        2634,\n        2635,\n        2636,\n        2637,\n        2638,\n        2639,\n        2640,\n        2641,\n        2642,\n        2643,\n        2644,\n        2645,\n        2646,\n        2647,\n        2648,\n        2649,\n        2650,\n        2651,\n        2652,\n        2653,\n        2654,\n        2655,\n        2656,\n        2657,\n        2658,\n        2659,\n        2660,\n        2661,\n        2662,\n        2663,\n        2664,\n        2665,\n        2666,\n        2667,\n        2668,\n        2669,\n        2670,\n        2671,\n        2672,\n        2673,\n        2674,\n        2675,\n        2676,\n        2677,\n        2678,\n        2679,\n        2680,\n        2681,\n        2682,\n        2683,\n        2684,\n        2685,\n        2686,\n        2687,\n        2688,\n        2689,\n        2690,\n        2691,\n        2692,\n        2693,\n        2694,\n        2695,\n        2696,\n        2697,\n        2698,\n        2699,\n        2700,\n        2701,\n        2702,\n        2703,\n        2704,\n        2705,\n        2706,\n        2707,\n        2708,\n        2709,\n        2710,\n        2711,\n        2712,\n        2713,\n        2714,\n        2715,\n        2716,\n        2717,\n        2718,\n        2719,\n        2720,\n        2721,\n        2722,\n        2723,\n        2724,\n        2725,\n        2726,\n        2727,\n        2728,\n        2729,\n        2730,\n        2731,\n        2732,\n        2733,\n        2734,\n        2735,\n        2736,\n        2737,\n        2738,\n        2739,\n        2740,\n        2741,\n        2742,\n        2743,\n        2744,\n        2745,\n        2746,\n        2747,\n        2748,\n        2749,\n        2750,\n        2751,\n        2752,\n        2753,\n        2754,\n        2755,\n        2756,\n        2757,\n        2758,\n        2759,\n        2760,\n        2761,\n        2762,\n        2763,\n        2764,\n        2765,\n        2766,\n        2767,\n        2768,\n        2769,\n        2770,\n        2771,\n        2772,\n        2773,\n        2774,\n        2775,\n        2776,\n        2777,\n        2778,\n        2779,\n        2780,\n        2781,\n        2782,\n        2783,\n        2784,\n        2785,\n        2786,\n        2787,\n        2788,\n        2789,\n        2790,\n        2791,\n        2792,\n        2793,\n        2794,\n        2795,\n        2796,\n        2797,\n        2798,\n        2799,\n        2800,\n        2801,\n        2802,\n        2803,\n        2804,\n        2805,\n        2806,\n        2807,\n        2808,\n        2809,\n        2810,\n        2811,\n        2812,\n        2813,\n        2814,\n        2815,\n        2816,\n        2817,\n        2818,\n        2819,\n        2820,\n        2821,\n        2822,\n        2823,\n        2824,\n        2825,\n        2826,\n        2827,\n        2828,\n        2829,\n        2830,\n        2831,\n        2832,\n        2833,\n        2834,\n        2835,\n        2836,\n        2837,\n        2838,\n        2839,\n        2840,\n        2841,\n        2842,\n        2843,\n        2844,\n        2845,\n        2846,\n        2847,\n        2848,\n        2849,\n        2850,\n        2851,\n        2852,\n        2853,\n        2854,\n        2855,\n        2856,\n        2857,\n        2858,\n        2859,\n        2860,\n        2861,\n        2862,\n        2863,\n        2864,\n        2865,\n        2866,\n        2867,\n        2868,\n        2869,\n        2870,\n        2871,\n        2872,\n        2873,\n        2874,\n        2875,\n        2876,\n        2877,\n        2878,\n        2879,\n        2880,\n        2881,\n        2882,\n        2883,\n        2884,\n        2885,\n        2886,\n        2887,\n        2888,\n        2889,\n        2890,\n        2891,\n        2892,\n        2893,\n        2894,\n        2895,\n        2896,\n        2897,\n        2898,\n        2899,\n        2900,\n        2901,\n        2902,\n        2903,\n        2904,\n        2905,\n        2906,\n        2907,\n        2908,\n        2909,\n        2910,\n        2911,\n        2912,\n        2913,\n        2914,\n        2915,\n        2916,\n        2917,\n        2918,\n        2919,\n        2920,\n        2921,\n        2922\n      ],\n      "train_dates": [\n        "2015-01-01 00:00:00+00:00",\n        "2021-12-25 00:00:00+00:00"\n      ],\n      "validation_dates": [\n        "2022-01-01 00:00:00+00:00",\n        "2022-12-31 00:00:00+00:00"\n      ],\n      "purge_days": 6\n    }\n  },\n  "seed": 4173,\n  "epochs": 24,\n  "batch_cases": 16,\n  "architecture": {\n    "features": 395,\n    "channels": 64,\n    "blocks": 4,\n    "convolutions_per_block": 2,\n    "kernel": 3,\n    "dilations": [\n      1,\n      2,\n      4,\n      8\n    ],\n    "padding": "case-local replication",\n    "hidden_normalization": "per-hour LayerNorm over64channels",\n    "activation": "ReLU",\n    "dropout": 0.1,\n    "head": "1x1 convolution to14logits",\n    "parameter_count": 126094,\n    "convolutional_receptive_field_hours": 61\n  },\n  "training": {\n    "optimizer": "AdamW",\n    "learning_rate": 0.001,\n    "weight_decay": 0.0001,\n    "betas": [\n      0.9,\n      0.999\n    ],\n    "eps": 1e-08,\n    "foreach": false,\n    "gradient_norm_clip": 1.0,\n    "schedule": "CosineAnnealingLR T_max24 eta_min.0001 stepped once after each epoch",\n    "loss": "unweighted mean BCEWithLogitsLoss",\n    "float32": true,\n    "amp": false,\n    "compile": false,\n    "early_stopping": null,\n    "checkpoint": "fixed epoch24 only",\n    "cpu_threads": 2,\n    "cuda_device": "cuda:0 only; no DataParallel or CPU fallback"\n  },\n  "normalization": "training cases only, per feature mean and population std; std<1e-6 replaced by1; clip normalized values to[-10,10]; saved immutable float64stats; transformedfloat32",\n  "anchor": {\n    "run_id": "lp-joint-comparison-001",\n    "analysis_sha256": "6b3ae133bcc8cb0b32edaaf45d4b8937f043a3bbdc52820a4b39ff338626bf13",\n    "source_order": [\n      "lgb-normalized-lp-001",\n      "lgb-normalized-lp-context-001",\n      "lgb-normalized-lp-joint-001"\n    ],\n    "weights": [\n      0.1875,\n      0.5625,\n      0.25\n    ],\n    "selection_sha256": "787cf5d98dd77afdf393dbb3dbd5fbec94ddfa1836b08a1b27c458cf8cec7359"\n  },\n  "blend_neural_weights": [\n    0.0,\n    0.1,\n    0.25,\n    0.5\n  ],\n  "selection": "Local only: choose2021 maximum pooled micro AUC, ties lower neural weight, then freeze before separate2022 command; retain accepted joint anchor if2022 fails improvement. No public-leaderboard tuning.",\n  "postprocessing": "Apply fixed Vinje projection after whole14-unit blend; no smoothing. Anchor arrays remain local.",\n  "remote_flow": "Default synthetic CUDA smoke only. Separate explicit --mode fit --year2021 and --year2022 invocations;2022 requires external local selection-lock file plus reviewed SHA256.",\n  "privacy": "Root owns private upload/session actions; this file itself performs no network or remote execution."\n}'
PROTOCOL = json.loads(PROTOCOL_TEXT)
PROTOCOL_SHA = hashlib.sha256(PROTOCOL_TEXT.encode()).hexdigest()
CODE_ROOT = Path('/kaggle/working') / ('tdt4173-tcn-code-' + PROTOCOL_SHA[:12])
CODE_ROOT.mkdir(parents=True, exist_ok=True)
for name, text in [('train_cuda_tcn.py', TRAINER_TEXT), ('protocol.json', PROTOCOL_TEXT)]:
    path = CODE_ROOT / name
    if path.exists():
        assert path.read_text() == text, 'Existing code differs; do not overwrite'
    else:
        path.write_text(text)
assert hashlib.sha256((CODE_ROOT / 'train_cuda_tcn.py').read_bytes()).hexdigest() == PROTOCOL['trainer_sha256']
uv = shutil.which('uv')
if uv is None:
    raise RuntimeError('uv is absent. No installation/download attempted; report this before continuing.')
ENV_ROOT = Path('/kaggle/working/tdt4173-cuda-system-env')
if not ENV_ROOT.exists():
    subprocess.run([uv, 'venv', '--offline', '--cache-dir', '/kaggle/working/tdt4173-uv-cache', '--system-site-packages', '--python', sys.executable, str(ENV_ROOT)], check=True)
assert 'include-system-site-packages = true' in (ENV_ROOT / 'pyvenv.cfg').read_text().lower()
PYTHON = str(ENV_ROOT / 'bin/python')
print('Code:', CODE_ROOT, 'protocol SHA256:', PROTOCOL_SHA)
print('uv environment reuses preinstalled CUDA packages; no package installs.')


In [ ]:
# Discover the actual read-only dataset mount; do not assume its name.
matches = []
for candidate in Path('/kaggle/input').rglob('bundle_manifest.json'):
    if hashlib.sha256(candidate.read_bytes()).hexdigest() == PROTOCOL['bundle_manifest_sha256']:
        matches.append(candidate)
if len(matches) != 1:
    raise RuntimeError(f'Expected one hash-matching private input bundle, found {len(matches)}')
BUNDLE_ROOT = matches[0].parent
manifest = json.loads(matches[0].read_text())
print('Read-only bundle mount:', BUNDLE_ROOT)
print('Payload files:', len(manifest['payload_files']), 'declared bytes:', sum(x['size_bytes'] for x in manifest['payload_files']))
# The real fold command separately stages and verifies every listed byte
# before importing/building frozen features. No feature table is built here.


In [ ]:
# This is the only compute command enabled by default. Synthetic data only.
SMOKE_RUN_ID = 'cuda-tcn-smoke-001'  # choose a fresh ID before repeating
subprocess.run([PYTHON, str(CODE_ROOT / 'train_cuda_tcn.py'), '--mode', 'smoke', '--protocol', str(CODE_ROOT / 'protocol.json'), '--run-id', SMOKE_RUN_ID], check=True)
SMOKE_MANIFEST = Path('/kaggle/working/tcn-runs') / SMOKE_RUN_ID / 'manifest.json'
print(json.loads(SMOKE_MANIFEST.read_text()))


## Separate reviewed fold commands

Both cells below are disabled. A normal Run All performs only the synthetic smoke. Activate one fold cell explicitly after review. Download all files in its output directory. The 2021 output must be selected locally against the frozen accepted joint anchor before2022 can begin. Anchor arrays are never uploaded.


In [ ]:
# DISABLED: after source review and a successful smoke, explicitly set True
# and run THIS cell only. Never run both real folds automatically.
RUN_2021_EXPLICITLY = False
if RUN_2021_EXPLICITLY:
    subprocess.run([PYTHON, str(CODE_ROOT / 'train_cuda_tcn.py'), '--mode', 'fit', '--year', '2021', '--protocol', str(CODE_ROOT / 'protocol.json'), '--run-id', 'cuda-tcn-2021-001', '--bundle', str(BUNDLE_ROOT), '--stage-root', '/kaggle/working/tdt4173-tcn-inputs-2021', '--smoke-manifest', str(SMOKE_MANIFEST)], check=True)
else:
    print('2021 real training remains disabled.')


The2022 lock must have version `cuda_tcn_2021_blend_lock_v1`, `selection_year:2021`, the current `training_protocol_sha256`, anchor run/hash from the protocol, `candidate_neural_weights:[0,0.1,0.25,0.5]`, a selected weight, `training_2021_complete:true`, and SHA256 values for the2021 neural run manifest and OOF file. The explicit CLI lock hash must match its bytes.


In [ ]:
# DISABLED: first download2021 OOF/checkpoint/manifest, select the fixed
# blend locally, then provide that reviewed lock file and its exact SHA256.
RUN_2022_EXPLICITLY = False
SELECTION_LOCK = Path('/kaggle/input/REVIEWED-LOCAL-LOCK/selection_2021.json')
SELECTION_LOCK_SHA256 = 'REPLACE_WITH_REVIEWED_LOCAL_SHA256'
if RUN_2022_EXPLICITLY:
    subprocess.run([PYTHON, str(CODE_ROOT / 'train_cuda_tcn.py'), '--mode', 'fit', '--year', '2022', '--protocol', str(CODE_ROOT / 'protocol.json'), '--run-id', 'cuda-tcn-2022-001', '--bundle', str(BUNDLE_ROOT), '--stage-root', '/kaggle/working/tdt4173-tcn-inputs-2022', '--smoke-manifest', str(SMOKE_MANIFEST), '--selection-lock', str(SELECTION_LOCK), '--selection-lock-sha256', SELECTION_LOCK_SHA256], check=True)
else:
    print('2022 real training remains disabled pending the external2021 selection lock.')
